# Évaluation Oracle — Phases 6 → 12

**Notebook unique** pour évaluer Oracle contre le baseline CorrDiff, avec interprétabilité étendue.

**Sept phases successives** :

1. **Phase 6 — Comparaison in-distribution standardisée** (post-hoc)

2. **Phase 7 — OOD réel sur EC-Earth3 + NorESM2-MM (6 runs alignés)**
   - Indices Rampal vendored : CDD, RX1day, R10mm, DJF/JJA, PSD distance
   - Sidecar probabiliste : CRPS, RMSE, spread, CRPS-SS, rank histogram, spread/skill ratio
   - K=12 ensemble members

3. **Phase 8 — Interprétabilité visuelle (8 figures)** : DAG, alpha, do(·), sensibilité, ablation A_dag, comparaison spatiale, RAPSD, histogramme

4. **Phase 9 — Diagnostics climate-standard** : Q-Q tail, Return-period Gumbel, Reliability, FSS

5. **Phase 10 — Cartes spatiales de biais d extrêmes** : 4 indices ETCCDI × 3 GCMs

6. **Phase 11 — Causalité forte** : DAG appris vs DAG physique (Q_phys) + Path-Specific Effects

7. **Phase 12 — Attribution rigoureuse** : Integrated Gradients (Sundararajan 2017, baseline climato Mamalakis 2022)

**Pré-requis** : checkpoints Oracle + CorrDiff sur Drive ; Cell 4 bootstrap autonome charge tout.

**Sortie** (post-Phase F) : `/content/drive/MyDrive/climate_data/results/oracle_evaluation/`  
**Baseline Oracle** (intacte) : `/content/drive/MyDrive/climate_data/results/v5_evaluation/`

In [ ]:
# >>> COLAB_BOOTSTRAP
# Bootstrap Colab optimisé — premier run ~3 min, re-runs ~30 s.
# Stratégie :
#   • Code sur SSD local (/content/) — git clone 5-10× plus rapide que vers Drive.
#   • Drive UNIQUEMENT pour les checkpoints (cf. cellule helpers plus bas).
#   • Pas de ``pip install -r requirements.txt`` brut (déclenche la compilation
#     CUDA de torch-scatter/torch-sparse → 20-30 min). À la place : install
#     pinned des seules deps non pré-installées par Colab.
#   • Wheels PyG pré-construits via le bon index (sinon torch-scatter/torch-sparse
#     compilent depuis les sources — interdit ici).
#
# Hors Colab : no-op.
import os, sys, subprocess, time, shlex
from pathlib import Path

# ── Configuration utilisateur ──────────────────────────────────────────
GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "two-stage-causal"  # hyperplan v2.0  # Phase 1 EDM rewrite (Karras 2022)
LOCAL_PROJECT = "/content/climate_data"  # SSD — toujours rapide
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True  # si ``import st_cdgm`` réussit déjà → skip pip

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    """Exécute une commande shell avec timing visible."""
    print(f"$ {cmd}")
    t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0
    print(f"  ↳ rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc


if _IS_COLAB:
    _T0 = time.time()
    print("🛰️  Colab détecté — bootstrap en cours…\n")

    # 1) Monter Drive (idempotent — pour la cellule de persistance plus loin)
    from google.colab import drive  # type: ignore[import-not-found]
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("   /content/drive déjà monté.")

    # 2) Clone vers SSD (PAS vers Drive — FUSE est lent pour des milliers de petits fichiers)
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        if GIT_URL is None:
            raise RuntimeError(
                "GIT_URL=None et projet absent du SSD. Renseignez GIT_URL ci-dessus, "
                "ou pré-uploadez le projet à " + LOCAL_PROJECT + " avant ce run."
            )
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")
    elif GIT_PULL_ON_RESUME:
        try:
            _run(f"git -C {LOCAL_PROJECT} pull --ff-only", timeout=60, check=False)
        except Exception as e:
            print(f"   ⚠️  git pull a levé : {e}")

    # 3) cd dans la racine — config/, src/, etc. en chemins relatifs
    os.chdir(project_path)
    print(f"   chdir → {os.getcwd()}\n")

    # 4) Test d'import — si st_cdgm marche déjà, on saute pip (énorme gain au re-run)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    _need_pip = True
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            _need_pip = False
            print("✓ Imports critiques OK — pip install sauté.")
        except ImportError as _imp_err:
            print(f"   import st_cdgm a échoué ({_imp_err}) — pip install requis.")

    if _need_pip:
        # 5) Versions PyTorch / CUDA déjà installées par Colab
        import torch
        TORCH_VER = torch.__version__.split("+")[0]  # ex. "2.5.1"
        TORCH_TAG = f"torch-{TORCH_VER}"             # ex. "torch-2.5.1"
        CUDA_TAG = "cu" + (torch.version.cuda or "121").replace(".", "") if torch.cuda.is_available() else "cpu"
        print(f"   torch={TORCH_VER}, cuda={CUDA_TAG}\n")

        # 6) Install des deps NON pré-installées par Colab
        # (numpy/pandas/scipy/sklearn/matplotlib/torch/torchvision/torchaudio/
        #  zarr/dask/xarray/h5netcdf/cartopy/tqdm/requests/ipykernel sont déjà là)
        EXTRA_DEPS = [
            "omegaconf==2.3.0",
            "hydra-core==1.3.2",
            "diffusers==0.36.0",
            "transformers==4.57.6",
            "accelerate==1.12.0",
            "huggingface-hub==0.36.0",
            "safetensors==0.7.0",
            "xbatcher",
            "webdataset",
            "cftime",
            "h5netcdf",
            "numcodecs",
            "torch-geometric",  # v2.3+ ne nécessite plus torch-scatter/torch-sparse
            "xformers",  # OOM fix R2 : memory-efficient attention sur sm_75 (T4)
        ]
        deps_str = " ".join(shlex.quote(p) for p in EXTRA_DEPS)
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location {deps_str}",
            timeout=600,
        )

        # 7) torch-scatter / torch-sparse — *optionnels* avec PyG ≥ 2.3 (fallback
        # pure-PyTorch). Décommentez si vous tombez sur un module qui les exige.
        # _PYG_INDEX = f"https://data.pyg.org/whl/{TORCH_TAG}+{CUDA_TAG}.html"
        # _run(
        #     f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
        #     f"torch-scatter torch-sparse -f {_PYG_INDEX}",
        #     timeout=600, check=False,
        # )

        # 8) Editable install du package — --no-deps pour ne PAS retomber sur
        # requirements.txt (qui réinstallerait torch et compilerait torch-scatter).
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
            f"--no-deps -e {LOCAL_PROJECT}",
            timeout=120,
        )

        # 9) Re-test des imports critiques
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            print("✓ st_cdgm importable.")
        except ImportError as e:
            print(f"⚠️  st_cdgm pas encore importable depuis ce kernel : {e}")
            print("   → Probablement un cache d'import — Runtime → Restart runtime puis re-run.")

    print(f"\n✅ Bootstrap Colab terminé en {time.time() - _T0:.1f}s.")

else:
    # Hors Colab : remonte automatiquement à la racine projet.
    _here = Path.cwd()
    for _candidate in [_here, *_here.parents]:
        if (_candidate / "config" / "training_config.yaml").exists() and (_candidate / "setup.py").exists():
            if _candidate != _here:
                os.chdir(_candidate)
                print(f"📂 chdir → {os.getcwd()} (racine projet détectée)")
            break
    print("ℹ️  Hors Colab — bootstrap sauté (assume install déjà faite).")

# === V5 specific smoke test (apres bootstrap) ===
try:
    from st_cdgm.models import ConditionalSkipBlock
    print("[Oracle smoke] ConditionalSkipBlock importable - Oracle features pretes")
except ImportError as e:
    print(f"[Oracle smoke] ConditionalSkipBlock NON disponible : {e}")
    print("           Verifier que la branche two-stage-causal contient src/st_cdgm/models/skip_direct.py")

try:
    from scripts.intervention_test import INTERVENTIONS
    print(f"[Oracle smoke] Phase 8 helpers OK ({len(INTERVENTIONS)} interventions)")
except ImportError as e:
    print(f"[Oracle smoke] scripts.intervention_test NON disponible : {e}")


In [ ]:
# === CONFIGURATION DES CHEMINS ===
from pathlib import Path

# Oracle (directory historiquement nomme ckpt_v2_corrdiff_normal)
V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")

# Noncausal = baseline CorrDiff générique
NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")

# Sorties
# Phase F (post-Oracle baseline, 2026-06-10) : nouveau dossier pour ne pas ecraser le baseline.
RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/oracle_evaluation")
V5_BASELINE_RESULTS_DIR = Path("/content/drive/MyDrive/climate_data/results/v5_evaluation")  # baseline intact

# Phase F (2026-06-11) : nouveau dir checkpoint pour preserver Oracle baseline
ORACLE_FINETUNED_DIR = Path("/content/drive/MyDrive/climate_data/oracle_finetuned")
# Selector : baseline = lit V5_DIR (intact) ; finetuned = lit ORACLE_FINETUNED_DIR
EVAL_VERSION = "baseline"  # change a "finetuned" apres Phase F pour evaluer le nouveau modele
ORACLE_DIR = ORACLE_FINETUNED_DIR if EVAL_VERSION == "finetuned" else V5_DIR
print(f"[Phase F] EVAL_VERSION = {EVAL_VERSION} -> ORACLE_DIR = {ORACLE_DIR.name}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# === Phase G (post-Oracle baseline, 2026-06-09) — selecteur de checkpoint ===
# "epoch_last" = Oracle baseline (gel §10.8 du memoire)
# "epoch_finetuned" = checkpoint post-Phase F (Bundle B + CASTLE + G_phys)
CHECKPOINT_NAME = "epoch_last"   # change a "epoch_finetuned" pour evaluer post-train
print(f"[Phase G] CHECKPOINT_NAME = {CHECKPOINT_NAME}")

# Vérification existence
# Verifie existence des checkpoints actifs (selon EVAL_VERSION et NONCAUSAL_DIR)
for label, p in [("Oracle", ORACLE_DIR), ("CorrDiff", NONCAUSAL_DIR)]:
    ckpt = p / f"{CHECKPOINT_NAME}.pth"
    fv = p / "final_validation_metrics.json"
    status_ckpt = "OK" if ckpt.exists() else "ABSENT"
    status_fv = "OK" if fv.exists() else "ABSENT"
    size_gb = ckpt.stat().st_size / 1e9 if ckpt.exists() else 0
    print(f"  {label:10s} : ckpt {status_ckpt} ({size_gb:.2f} GB)  metrics.json {status_fv}")
    print(f"             {p}")

print(f"\nRésultats -> {RESULTS_DIR}")

---

## Bootstrap autonome — CONFIG / pipeline / stacks

Cette cellule rend le notebook entièrement **autonome** : plus besoin d'exécuter les cellules du notebook training avant.

Elle :
1. Charge `CONFIG` (base + override `corrdiff_normal`)
2. Définit `DEVICE`, `lr_shape`, `hr_shape`
3. Construit le `NetCDFDataPipeline` ACCESS-CM2 (in-distribution)
4. Définit `builder` et `convert_sample_to_batch`
5. Charge les **deux stacks complets** : Oracle et CorrDiff (encoder + RCN + regression_head + skip_block + diffusion)
6. Définit `predict_with_stack()` réutilisé par toutes les phases

Coût : ~3 min (chargement Drive + checkpoints)

In [ ]:
# ============================================================
# Bootstrap autonome COMPLET (= Cells 15+16+17+30+32+40 du training notebook)
# Telecharge automatiquement TOUS les datasets manquants depuis Zenodo
# ============================================================
import os
import sys
import time
import json
import shutil
import urllib.request
import urllib.error
import torch
import numpy as np
from pathlib import Path
from omegaconf import OmegaConf

ON_COLAB = "google.colab" in sys.modules or Path("/content").exists()

# 1. CONFIG (base + override corrdiff_normal si dispo)
_base = Path("config/training_config.yaml")
_override = Path("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.load(_base)
if _override.exists():
    CONFIG = OmegaConf.merge(CONFIG, OmegaConf.load(_override))
    print("[OK] CONFIG = base + corrdiff_normal override")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
print(f"[OK] DEVICE={DEVICE}  lr_shape={lr_shape}  hr_shape={hr_shape}")

# 2. DATA_ROOT detection (= training cell 16)
DATA_ROOT_LOCAL = Path("data/raw")
DATA_ROOT_DRIVE = Path("/content/drive/MyDrive/climate_data/data")
_DATA_ROOT_LOCAL_SSD = Path("/content/data_local")

if ON_COLAB and DATA_ROOT_DRIVE.parent.parent.exists():
    DATA_ROOT = DATA_ROOT_DRIVE
    print(f"[INFO] DATA_ROOT = Drive ({DATA_ROOT})")
else:
    DATA_ROOT = DATA_ROOT_LOCAL
    print(f"[INFO] DATA_ROOT = local ({DATA_ROOT.resolve()})")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 3. BS32 SSD copy (Drive -> /content/data_local pour I/O rapide)
_BS32_ENABLED = bool(globals().get("DATA_LOCAL_SSD", True))
if ON_COLAB and _BS32_ENABLED and DATA_ROOT == DATA_ROOT_DRIVE:
    _files_to_copy = [
        ("train/predictor_ACCESS-CM2_hist.nc",   "predictor_ACCESS-CM2_hist.nc"),
        ("train/pr_ACCESS-CM2_hist.nc",          "pr_ACCESS-CM2_hist.nc"),
        ("static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc",
         "ERA5_eval_ccam_12km.198110_NZ_Invariant.nc"),
        ("normalization_coefs/mean_1974_2011.nc", "mean_1974_2011.nc"),
        ("normalization_coefs/std_1974_2011.nc",  "std_1974_2011.nc"),
    ]
    _ssd_train = _DATA_ROOT_LOCAL_SSD / "train"
    _ssd_static = _DATA_ROOT_LOCAL_SSD / "static_predictors"
    _ssd_norm = _DATA_ROOT_LOCAL_SSD / "normalization_coefs"
    for _d in (_ssd_train, _ssd_static, _ssd_norm):
        _d.mkdir(parents=True, exist_ok=True)
    _t_total = time.time()
    _bytes_copied = 0
    for _rel, _name in _files_to_copy:
        _src = DATA_ROOT_DRIVE / _rel
        if "train/" in _rel:
            _dst = _ssd_train / _name
        elif "static_predictors/" in _rel:
            _dst = _ssd_static / _name
        else:
            _dst = _ssd_norm / _name
        if not _src.exists():
            continue
        if _dst.exists() and _dst.stat().st_size == _src.stat().st_size:
            continue
        _t0 = time.time()
        print(f"   copie {_src.name}...", flush=True)
        shutil.copy2(_src, _dst)
        _bytes_copied += _dst.stat().st_size
        print(f"   OK {_dst.name} ({_dst.stat().st_size/1e6:.0f} MB en {time.time()-_t0:.1f}s)")
    if _bytes_copied > 0:
        print(f"BS32 SSD copy: {_bytes_copied/1e9:.2f} GB en {time.time()-_t_total:.1f}s")
    DATA_ROOT = _DATA_ROOT_LOCAL_SSD
    print(f"[INFO] DATA_ROOT redirige vers SSD : {_DATA_ROOT_LOCAL_SSD}")

# 4. Resolution des paths (= training cell 16 suite)
def _relocate(p):
    if not p:
        return p
    s = str(p)
    if s.startswith("data/raw/"):
        return str(DATA_ROOT / s[len("data/raw/"):])
    return s

for _key in ("lr_path", "hr_path", "static_path"):
    if CONFIG.data.get(_key):
        CONFIG.data[_key] = _relocate(CONFIG.data[_key])

LR_PATH = str(CONFIG.data.lr_path)
HR_PATH = str(CONFIG.data.hr_path)
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None
MEAN_PATH = str(DATA_ROOT / "normalization_coefs" / "mean_1974_2011.nc")
STD_PATH = str(DATA_ROOT / "normalization_coefs" / "std_1974_2011.nc")

URL_ZENODO_HR = "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1"
URL_ZENODO_LR = "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1"
URLS_TEST = [
    ("EC-Earth3_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_histupdated_compressed.nc?download=1"),
    ("EC-Earth3_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/EC-Earth3_historical_precip_compressed.nc?download=1"),
    ("NorESM2-MM_histupdated_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_histupdated_compressed.nc?download=1"),
    ("NorESM2-MM_historical_precip_compressed.nc",
     "https://zenodo.org/records/10889046/files/NorESM2-MM_historical_precip_compressed.nc?download=1"),
]

# 5. stream_download (atomique + reprise + timeout)
def stream_download(url, dest, retries=5, chunk_size=1024*1024,
                    connect_timeout=30, read_timeout=120):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = urllib.request.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
            print(f"   reprise a {already/1e6:.1f} MB")
        try:
            with urllib.request.urlopen(req, timeout=connect_timeout) as resp:
                total = resp.length
                if total is None and resp.headers.get("Content-Length"):
                    total = int(resp.headers["Content-Length"])
                grand_total = (total + already) if total else None
                mode = "ab" if already > 0 else "wb"
                with open(part, mode) as f:
                    downloaded = already
                    last_log = time.time()
                    last_log_bytes = downloaded
                    while True:
                        chunk = resp.read(chunk_size)
                        if not chunk:
                            break
                        f.write(chunk)
                        downloaded += len(chunk)
                        now = time.time()
                        if now - last_log >= 5.0:
                            speed = (downloaded - last_log_bytes) / (now - last_log) / 1e6
                            if grand_total:
                                pct = 100.0 * downloaded / grand_total
                                print(f"     {downloaded/1e6:7.1f}/{grand_total/1e6:7.1f} MB ({pct:.0f}%) {speed:.1f} MB/s")
                            else:
                                print(f"     {downloaded/1e6:7.1f} MB {speed:.1f} MB/s")
                            last_log = now
                            last_log_bytes = downloaded
            os.replace(part, dest)
            print(f"   OK {dest.name} ({dest.stat().st_size/1e6:.0f} MB)")
            return True
        except urllib.error.HTTPError as e:
            if e.code in (503, 504, 429):
                wait = min(60, 2**attempt); print(f"   HTTP {e.code} retry {wait}s"); time.sleep(wait)
            elif e.code == 416:
                os.replace(part, dest); return True
            else:
                print(f"   HTTP {e.code}: {e.reason}"); return False
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2**attempt); print(f"   reseau retry {wait}s ({type(e).__name__})"); time.sleep(wait)
        except Exception as e:
            print(f"   ERREUR {type(e).__name__}: {e}"); return False
    return False

# 6. Telechargements train (HR, LR ACCESS-CM2)
if not Path(HR_PATH).exists():
    print(f"Download HR ACCESS-CM2: {HR_PATH}")
    if not stream_download(URL_ZENODO_HR, HR_PATH):
        raise RuntimeError("Echec download HR ACCESS-CM2")
if not Path(LR_PATH).exists():
    print(f"Download LR ACCESS-CM2: {LR_PATH}")
    if not stream_download(URL_ZENODO_LR, LR_PATH):
        raise RuntimeError("Echec download LR ACCESS-CM2")

# 7. Telechargements test (EC-Earth3, NorESM2-MM)
TEST_ROOT = DATA_ROOT / "test"
TEST_ROOT.mkdir(parents=True, exist_ok=True)
for _filename, _url in URLS_TEST:
    _filepath = TEST_ROOT / _filename
    if _filepath.exists():
        continue
    print(f"Download test: {_filename}")
    if not stream_download(_url, str(_filepath)):
        raise RuntimeError(f"Echec download {_filename}")

# 8. Statics + normalization (fallback gdown si Drive public)
_PUBLIC_DRIVE_FALLBACKS = {
    "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc":
        "1KY6IS1W5Wt-l_xyV7Qw8caA49zPzuSEx",
    "normalization_coefs/mean_1974_2011.nc":
        "14wVaJTUDgLwLlFcqRFA6pzJg9tZtAVQ0",
    "normalization_coefs/std_1974_2011.nc":
        "1ycqq9DqpfdOOiyQqgKs797OzRdHND3ZL",
}

def _gdown_install():
    try:
        import gdown; return True
    except ImportError:
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"], timeout=120)
            import gdown; return True
        except Exception:
            return False

def _try_gdown(path):
    if not path:
        return False
    pth = Path(path)
    rel_key = None
    for _key in _PUBLIC_DRIVE_FALLBACKS:
        if str(pth).endswith(_key.replace("/", os.sep)) or str(pth).endswith(_key):
            rel_key = _key; break
    if rel_key is None:
        return False
    file_id = _PUBLIC_DRIVE_FALLBACKS[rel_key]
    pth.parent.mkdir(parents=True, exist_ok=True)
    if not _gdown_install():
        return False
    import gdown
    try:
        print(f"   gdown.download(id={file_id}) -> {pth}")
        gdown.download(id=file_id, output=str(pth), quiet=False)
        return pth.exists() and pth.stat().st_size > 0
    except Exception as e:
        print(f"   gdown ERREUR: {e}"); return False

for _var, _name in [("STATIC_PATH", "Static"), ("MEAN_PATH", "Mean"), ("STD_PATH", "Std")]:
    _p = globals()[_var]
    if _p and Path(_p).exists():
        continue
    if _try_gdown(_p):
        print(f"   OK {_name} (gdown public)")
    else:
        print(f"   {_name} absent: {_p} -> None")
        globals()[_var] = None

# 9. Resume final
print()
print("Datasets disponibles :")
print(f"  LR train  : {LR_PATH}  ({'OK' if Path(LR_PATH).exists() else 'MISSING'})")
print(f"  HR train  : {HR_PATH}  ({'OK' if Path(HR_PATH).exists() else 'MISSING'})")
print(f"  Static    : {STATIC_PATH}  ({'OK' if STATIC_PATH and Path(STATIC_PATH).exists() else 'NONE'})")
print(f"  Mean/Std  : {MEAN_PATH} / {STD_PATH}")
for fname, _ in URLS_TEST:
    p = TEST_ROOT / fname
    print(f"  Test      : {p.name}  ({'OK' if p.exists() else 'MISSING'})")

# 10. GCM_REGISTRY pour OOD
GCM_REGISTRY = {
    "ACCESS-CM2":  (Path(LR_PATH), Path(HR_PATH), True),
    "EC-Earth3":   (TEST_ROOT / "EC-Earth3_histupdated_compressed.nc",
                     TEST_ROOT / "EC-Earth3_historical_precip_compressed.nc", False),
    "NorESM2-MM":  (TEST_ROOT / "NorESM2-MM_histupdated_compressed.nc",
                     TEST_ROOT / "NorESM2-MM_historical_precip_compressed.nc", False),
}

# 11. Pipeline + builder
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder

def make_pipeline(lr_path, hr_path):
    return NetCDFDataPipeline(
        lr_path=str(lr_path), hr_path=str(hr_path),
        static_path=str(STATIC_PATH) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        seq_len=int(CONFIG.data.seq_len),
        baseline_strategy=str(CONFIG.data.baseline_strategy),
        baseline_factor=int(CONFIG.data.baseline_factor),
        target_transform=str(CONFIG.data.get("target_transform", "log1p")),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
        precipitation_delta=float(CONFIG.data.get("precipitation_delta", 0.01)),
        lr_variables=list(CONFIG.data.lr_variables),
        hr_variables=list(CONFIG.data.hr_variables),
        static_variables=list(CONFIG.data.static_variables) if STATIC_PATH and Path(STATIC_PATH).exists() else None,
        means_path=str(MEAN_PATH) if MEAN_PATH and Path(MEAN_PATH).exists() else None,
        stds_path=str(STD_PATH) if STD_PATH and Path(STD_PATH).exists() else None,
        eager_load_datasets=bool(CONFIG.data.get("eager_load_datasets", False)),
    )

pipeline_access = make_pipeline(LR_PATH, HR_PATH)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline_access.get_static_dataset(),
    include_mid_layer=bool(CONFIG.graph.include_mid_layer),
)
print()
print(f"[OK] Builder cree ({len(builder.dynamic_node_types)} dyn + {len(builder.static_node_types)} static)")

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {"lr": lr_tensor, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero}

test_dataset = pipeline_access.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True,
)
sample = next(iter(test_dataset))
_runtime_dim = int(sample["lr"].shape[1])
if _runtime_dim != int(CONFIG.rcn.driver_dim):
    CONFIG.rcn.driver_dim = _runtime_dim
    CONFIG.rcn.reconstruction_dim = _runtime_dim
    print(f"[INFO] CONFIG.rcn.driver_dim -> {_runtime_dim}")

# 12. Stacks
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
from st_cdgm.models.edm_preconditioner import EDMConfig
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    ConditionalSkipBlock = None

def build_stack(ckpt_path, name):
    print(f"  [{name}] {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # 1. Encoder configs depuis CONFIG.encoder.metapaths (filtre selon allowed_nodes du builder)
    allowed_nodes = set(builder.dynamic_node_types + builder.static_node_types)
    encoder_configs = []
    for _mp in CONFIG.encoder.metapaths:
        _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
        if _src in allowed_nodes and _tgt in allowed_nodes:
            encoder_configs.append(IntelligibleVariableConfig(
                name=_mp.name,
                meta_path=(_src, _rel, _tgt),
                pool=_mp.get("pool", "mean"),
            ))
    # Ajoute le static metapath si pipeline a static_dataset
    if pipeline_access.get_static_dataset() is not None:
        encoder_configs.append(IntelligibleVariableConfig(
            name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
        ))

    enc = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
    ).to(DEVICE)
    num_vars = len(encoder_configs)

    rcn_cell = RCNCell(
        num_vars=num_vars,
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=int(CONFIG.rcn.driver_dim),
        reconstruction_dim=int(CONFIG.rcn.reconstruction_dim),
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

    rh = GraphToGridDecoder(
        d_model=CONFIG.encoder.hidden_dim,
        hr_h=CONFIG.graph.hr_shape[0], hr_w=CONFIG.graph.hr_shape[1],
    ).to(DEVICE)

    edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
    # Convertit unet_kwargs en dict + tuples pour block_types
    from omegaconf import OmegaConf as _OC
    _unet_kwargs = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for _k in ("down_block_types", "up_block_types"):
        if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
            _unet_kwargs[_k] = tuple(_unet_kwargs[_k])

    hr_channels = int(sample["residual"].shape[1])

    diff = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height),
        width=int(CONFIG.diffusion.width),
        unet_kwargs=_unet_kwargs,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", False)),
        conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
        anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
        edm_config=edm_cfg,
        causal_concat=True,
    ).to(DEVICE)
    def _safe_load(name, module):
        """Load state_dict avec checks None + dict-like + strip prefixes."""
        key = f"{name}_state_dict"
        if key not in ckpt:
            print(f"  [{name}] [WARN] {key} absent du checkpoint")
            return False
        sd = ckpt[key]
        if sd is None:
            print(f"  [{name}] [WARN] {key} is None - skip")
            return False
        if not hasattr(sd, "items"):
            print(f"  [{name}] [WARN] {key} not dict-like ({type(sd).__name__}) - skip")
            return False
        # Strip prefixes torch.compile ('_orig_mod.') et DDP ('module.')
        prefixes = ["_orig_mod.", "module."]
        stripped = {}
        for kk, vv in sd.items():
            new_k = kk
            for p in prefixes:
                if new_k.startswith(p):
                    new_k = new_k[len(p):]
            stripped[new_k] = vv
        try:
            missing, unexpected = module.load_state_dict(stripped, strict=False)
            if missing:
                print(f"  [{name}] [INFO] {len(missing)} keys manquantes (premieres : {missing[:3]})")
            if unexpected:
                print(f"  [{name}] [INFO] {len(unexpected)} keys inattendues (premieres : {unexpected[:3]})")
            return True
        except Exception as e:
            print(f"  [{name}] [ERREUR] load_state_dict: {type(e).__name__}: {e}")
            return False

    for n, m in [("encoder", enc), ("rcn_cell", rcn_cell),
                  ("regression_head", rh), ("diffusion", diff)]:
        _safe_load(n, m)
    skip = None
    if (SKIP_AVAILABLE and "skip_block_state_dict" in ckpt
            and ckpt["skip_block_state_dict"] is not None):
        skip = ConditionalSkipBlock(
            lr_channels=len(CONFIG.data.lr_variables),
            hr_shape=tuple(CONFIG.graph.hr_shape),
        ).to(DEVICE)
        try:
            skip.load_state_dict(ckpt["skip_block_state_dict"], strict=False)
            print(f"  [{name}] [+] skip_block ({skip.num_params()} params)")
        except Exception as e:
            print(f"  [{name}] [WARN] skip_block load failed: {e}")
            skip = None
    enc.eval(); rcn_cell.eval(); rh.eval(); diff.eval()
    if skip is not None:
        skip.eval()
    A_dag = rcn_cell.A_dag.detach().cpu().clone() if hasattr(rcn_cell, "A_dag") else None
    return {"encoder": enc, "rcn_runner": rcn_runner, "regression_head": rh,
            "diffusion": diff, "skip_block": skip, "A_dag": A_dag, "variant": name}

print()
print("Chargement des stacks...")
t0 = time.time()
# Phase F (2026-06-11) : ORACLE_DIR resolu via EVAL_VERSION (baseline / finetuned).
# Le baseline CorrDiff (NONCAUSAL_DIR) reste sur "epoch_last.pth" (jamais touche).
stack_v5 = build_stack(ORACLE_DIR / f"{CHECKPOINT_NAME}.pth", "Oracle")
stack_nc = build_stack(NONCAUSAL_DIR / "epoch_last.pth", "CorrDiff")
print(f"[OK] 2 stacks charges en {time.time()-t0:.1f}s")

# 13. Predict generique
@torch.no_grad()
def predict_with_stack(stack, batch, K=4, n_steps=32):
    enc, rcn, rh, diff, skip = (stack["encoder"], stack["rcn_runner"],
                                  stack["regression_head"], stack["diffusion"],
                                  stack["skip_block"])
    lr = batch["lr"].to(DEVICE)
    H_init = enc.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr[t] for t in range(lr.shape[0])]
    seq = rcn.run(H_init, drivers, reconstruction_sources=None)
    H_T = seq.states[-1]
    mu_c = rh(H_T)
    tshape = batch["residual"][-1].to(DEVICE).shape
    if tshape[-2:] != mu_c.shape[-2:]:
        mu_c = torch.nn.functional.interpolate(
            mu_c, size=tshape[-2:], mode="bilinear", align_corners=False,
        )
    if skip is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu, _ = skip(lr_last, mu_c)
    else:
        mu = mu_c
    mu = torch.nan_to_num(mu, nan=0.0)
    bl = batch["baseline"][-1].to(DEVICE)
    if bl.dim() == mu.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    ens = []
    for _ in range(K):
        o = diff.sample(
            conditioning=None, num_steps=n_steps,
            scheduler_type="edm_karras", apply_constraints=False,
            mu_HR=mu, baseline_log=bl,
        )
        r = o.residual if hasattr(o, "residual") else o
        ens.append((bl + mu + r).cpu())
    return torch.stack(ens, dim=0)

print()
print("=" * 70)
print("Bootstrap autonome COMPLET")
print("=" * 70)
print("Variables disponibles :")
print(f"  CONFIG, DEVICE, builder, convert_sample_to_batch, predict_with_stack")
print(f"  stack_v5, stack_nc, GCM_REGISTRY, make_pipeline")
print(f"  test_dataset (ACCESS-CM2 in-dist)")
print()
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"TEST_ROOT  : {TEST_ROOT}")


---

## Phase F — Fine-tune Bundle B + CASTLE + G_phys (Oracle, 2026-06-11)

**Cellule optionnelle** qui re-entraine `stack_v5` (encoder + RCN + reg_head + skip si dispo)
avec toutes les modifs Phases A-E :

- **CASTLE-style joint prediction anchoring** sur A_dag (Phase A)
- **Weighted MSE power-law** + **L1 cosine annealing** (Phase B)
- **Pinball quantile loss** (τ=0.95, 0.99) + **CC regularizer** + **spectral high-k** (Phase C)
- **Masque physique G_phys** (Phase D) — couplage QG descendant
- **TailStratifiedSampler** (Phase E) — 30 % extrêmes garantis par batch

Stage 2 (diffusion) est **figé**. `sigma_data` est **recalibré** en fin de run.

Pour éviter de claquer 4h en aveugle, le code expose un flag `SMOKE_TEST` :
- `SMOKE_TEST = True`  → 2 epochs (~20-30 min sur GPU T4, valide la pipeline)
- `SMOKE_TEST = False` → 25 epochs (~30-50 min GPU T4, ~3-4h CPU) — le vrai run

**Une fois le fine-tune termine**, change `EVAL_VERSION = "finetuned"` en Cell 2 et
re-execute les cellules a partir de Cell 4 pour evaluer le nouveau modele.

Voir `architecture_journey.md` §12 pour le plan complet et les cibles d'amelioration.


In [ ]:
# === Phase F : fine-tune Bundle B + CASTLE + G_phys ===
# Met SMOKE_TEST = True pour un test rapide 2 epochs avant le vrai run.
SMOKE_TEST = False   # 25 epochs full run + recompute Phase 6 a la fin

EPOCHS = 2 if SMOKE_TEST else 25
SANITY_EVERY = 1 if SMOKE_TEST else 5
# === Pre-flight checks ===
import os, shutil, psutil, gc

def _gb(b):
    return b / (1024**3)

def _phase_f_preflight():
    issues = []
    # RAM
    ram = psutil.virtual_memory()
    print(f"  RAM dispo : {_gb(ram.available):.1f} GB / {_gb(ram.total):.1f} GB total")
    if _gb(ram.available) < 12.0:
        issues.append(f"RAM dispo < 12 GB ({_gb(ram.available):.1f} GB) — risque OOM")
    # Disque Drive
    try:
        du = shutil.disk_usage("/content/drive/MyDrive")
        print(f"  Disque Drive : {_gb(du.free):.1f} GB libre / {_gb(du.total):.1f} GB total")
        if _gb(du.free) < 5.0:
            issues.append(f"Drive < 5 GB libre — risque saturation checkpoints")
    except Exception as e:
        print(f"  [WARN] disk check failed: {e}")
    # Detect smoke-test artifact
    smoke_ckpt = ORACLE_FINETUNED_DIR / "epoch_finetuned.pth"
    inprogress = ORACLE_FINETUNED_DIR / "epoch_finetuned_inprogress.pth"
    if smoke_ckpt.exists() and not inprogress.exists():
        # Detect smoke test (file present but no inprogress = a previous run finished)
        try:
            import torch as _t
            _ck = _t.load(smoke_ckpt, map_location="cpu", weights_only=False)
            saved_ep = _ck.get("epoch", "?")
            if isinstance(saved_ep, int) and saved_ep < EPOCHS:
                print(f"  [INFO] Smoke test ancien detecte : {smoke_ckpt.name} (epoch={saved_ep})")
                print(f"         Sera ecrase au final save (epoch_finetuned.pth)")
        except Exception:
            pass
    if issues:
        print()
        print("  [WARNING] Issues detectees :")
        for it in issues:
            print(f"    - {it}")
        print()
    return issues

print()
print("[Phase F] Pre-flight check...")
_phase_f_preflight()
print()
# === Setup ORACLE_FINETUNED_DIR : copie le baseline Oracle comme point de depart ===
ORACLE_FINETUNED_DIR.mkdir(parents=True, exist_ok=True)
_baseline_ckpt = V5_DIR / "epoch_last.pth"
_oracle_ft_ckpt = ORACLE_FINETUNED_DIR / "epoch_last.pth"
if not _oracle_ft_ckpt.exists():
    print(f"[Setup] Copie du baseline Oracle vers oracle_finetuned/ (premiere execution)...")
    import shutil as _sh
    _sh.copy(_baseline_ckpt, _oracle_ft_ckpt)
    _size_gb = _oracle_ft_ckpt.stat().st_size / 1024**3
    print(f"  [OK] {_oracle_ft_ckpt.name} ({_size_gb:.2f} GB) copie depuis baseline Oracle")
    # Important : reload stack_v5 depuis le nouveau ckpt pour que finetune_bundle_b
    # modifie l'instance qui pointe vers oracle_finetuned/ et non vers V5_DIR
    print(f"  [Note] Le stack Oracle actuel pointe vers le baseline. Le fine-tune va")
    print(f"         modifier stack_v5 in-place et sauvegarder dans ORACLE_FINETUNED_DIR.")
else:
    print(f"[Setup] {_oracle_ft_ckpt.name} existe deja ({_oracle_ft_ckpt.stat().st_size/1024**3:.2f} GB)")
print()




from scripts.finetune_stage1_bundle_b import finetune_bundle_b
from torch.utils.data import Dataset as _TorchDataset


class _MapStyleListDataset(_TorchDataset):
    """Wrapper map-style autour d'une liste de samples materialises.

    TailStratifiedSampler et compute_sample_max_values necessitent __len__ et
    __getitem__, ce que ResDiffIterableDataset ne fournit pas.
    """
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        return self.samples[i]


# 1. Construire un train_dataset distinct du test_dataset (stride=1)
print(f"[Phase F] Construction du train_dataset (ACCESS-CM2)...")
pipe_train = make_pipeline(GCM_REGISTRY["ACCESS-CM2"][0], GCM_REGISTRY["ACCESS-CM2"][1])
train_iter = pipe_train.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len),
    stride=1,
    as_torch=True,
)

# 2. Materialize l'IterableDataset en liste pour usage map-style.
#    ATTENTION : build_sequence_dataset(stride=1) sur 20-30 ans d'ACCESS-CM2
#    genere ~10000 sequences (~100 GB RAM). On limite a MAX_TRAIN_SAMPLES
#    qui represente ~4 ans avec stride=1, largement suffisant pour fine-tune.
MAX_TRAIN_SAMPLES = 750    # ~7.5 GB RAM (~2 ans stride=1), reduire a 500 si OOM
MAX_VAL_SAMPLES = 150      # ~1.5 GB RAM, suffisant pour recalibration sigma_data

print(f"  Materialization train_dataset (limite a {MAX_TRAIN_SAMPLES} samples, ~1-2 min)...")
import time as _t
import itertools as _it
_t0 = _t.time()
_samples = list(_it.islice(train_iter, MAX_TRAIN_SAMPLES))
print(f"  [OK] {len(_samples)} samples materialises en {_t.time()-_t0:.1f}s")
train_dataset = _MapStyleListDataset(_samples)

# Idem pour test_dataset s'il est iterable
if not hasattr(test_dataset, "__len__"):
    print(f"  Materialization val_dataset (limite a {MAX_VAL_SAMPLES} samples)...")
    _val_samples = list(_it.islice(test_dataset, MAX_VAL_SAMPLES))
    val_dataset = _MapStyleListDataset(_val_samples)
    print(f"  [OK] {len(_val_samples)} val samples materialises")
else:
    val_dataset = test_dataset

print(f"  train_dataset : {len(train_dataset)} samples")
print(f"  val_dataset   : {len(val_dataset)} samples")
print(f"  SMOKE_TEST = {SMOKE_TEST}  ->  EPOCHS = {EPOCHS}")
print()

# === Pre-Phase F : recompute baselines avec formules alignees ===
# CRITICAL : sans cela, la comparaison Phase 6 post-finetune n'est pas
# apples-to-apples (baseline JSONs utilisent l'ancienne formule F1/RAPSD).
# On produit des _aligned.json pour Oracle baseline et CorrDiff, qui
# servent de reference dans Phase 6 (Cell 8).
from scripts.recompute_phase6_metrics import recompute_phase6_metrics as _recompute_aligned

_v5_aligned = V5_DIR / "final_validation_metrics_aligned.json"
_nc_aligned = NONCAUSAL_DIR / "final_validation_metrics_aligned.json"

if not _v5_aligned.exists():
    print()
    print("[Pre-recompute] Production des baseline JSONs avec formules alignees...")
    print("  (necessaire pour comparaison apples-to-apples avec post-Phase F)")
    try:
        _recompute_aligned(
            stack=stack_v5, builder=builder, val_dataset=val_dataset,
            DEVICE=DEVICE,
            predict_with_stack_fn=predict_with_stack,
            convert_sample_to_batch_fn=convert_sample_to_batch,
            out_path=_v5_aligned,
            K_samples=12, n_steps=18, n_batches=16,
            epoch=0,  # baseline epoch
            verbose=True,
        )
        print(f"  [OK] Oracle baseline aligne : {_v5_aligned.name}")
    except Exception as _e:
        warnings.warn(f"Pre-recompute Oracle baseline failed: {_e}")

if not _nc_aligned.exists():
    try:
        _recompute_aligned(
            stack=stack_nc, builder=builder, val_dataset=val_dataset,
            DEVICE=DEVICE,
            predict_with_stack_fn=predict_with_stack,
            convert_sample_to_batch_fn=convert_sample_to_batch,
            out_path=_nc_aligned,
            K_samples=12, n_steps=18, n_batches=16,
            epoch=0, verbose=False,
        )
        print(f"  [OK] CorrDiff aligne : {_nc_aligned.name}")
    except Exception as _e:
        warnings.warn(f"Pre-recompute CorrDiff failed: {_e}")
print()

import warnings  # safety
# 3. Lancer le fine-tune
result = finetune_bundle_b(
    stack=stack_v5,
    builder=builder,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    CONFIG=CONFIG,
    DEVICE=DEVICE,
    epochs=EPOCHS,
    batch_size=8,
    ckpt_save_dir=ORACLE_FINETUNED_DIR,
    convert_sample_to_batch_fn=convert_sample_to_batch,
    sanity_eval_every=SANITY_EVERY,
    seed=42,
    skip_sigma_data_recalib=SMOKE_TEST,   # skip recalib en smoke test
)

print()
print("=" * 60)
print(f"[OK] Phase F terminee  ({EPOCHS} epochs)")
print(f"  Checkpoint sauvegarde : {result['final_ckpt']}")
print(f"  Nouveau sigma_data    : {result['sigma_data_new']}")
print("=" * 60)
print()
# 4. Recompute Phase 6 metrics avec le stack fine-tune
if not SMOKE_TEST:
    print()
    print("=" * 60)
    print("[Phase 6 recompute] Recalcul des metrics avec stack fine-tune...")
    print("=" * 60)
    from scripts.recompute_phase6_metrics import recompute_phase6_metrics

    # V5_DIR (baseline Oracle) reste intact — pas d'overwrite/backup necessaire.
    # ORACLE (post-finetune) — ecrit dans ORACLE_FINETUNED_DIR (avec try/except safety)
    try:
        recompute_phase6_metrics(
        stack=stack_v5, builder=builder, val_dataset=val_dataset,
        DEVICE=DEVICE,
        predict_with_stack_fn=predict_with_stack,
        convert_sample_to_batch_fn=convert_sample_to_batch,
        out_path=ORACLE_FINETUNED_DIR / "final_validation_metrics.json",
        K_samples=12, n_steps=18, n_batches=16,
        epoch=EPOCHS, causal_concat=True,
    )
        print()
        print("[OK] ORACLE_FINETUNED_DIR/final_validation_metrics.json mis a jour")
        print()
    except Exception as _e:
        print(f"[WARN] Recompute Phase 6 a echoue : {type(_e).__name__}: {_e}")
        print(f"       Le checkpoint training reste valide (epoch_finetuned.pth)")
        print(f"       Tu peux relancer le recompute manuellement plus tard")
        print()

# Memory cleanup : free 9-10 GB avant Phases 7-12
print()
print("[Cleanup] Liberation memoire post-training...")
try:
    del _samples
except NameError:
    pass
try:
    del _val_samples
except NameError:
    pass
try:
    del train_iter, train_dataset, val_dataset
except NameError:
    pass
import gc as _gc
_gc.collect()
if 'torch' in dir() and torch.cuda.is_available():
    torch.cuda.empty_cache()
import psutil as _psutil
print(f"  [OK] RAM apres cleanup : {_psutil.virtual_memory().available / 1024**3:.1f} GB dispo")
print()

print("PROCHAINE ETAPE :")
print("  1. Si SMOKE_TEST = True : passe a SMOKE_TEST = False et relance cette cellule")
print("  2. Une fois fine-tune complet : change EVAL_VERSION = \"finetuned\" en Cell 2")
print("  3. Re-execute Cell 4 (recharge stack depuis ORACLE_FINETUNED_DIR), puis Cells 8, 10, 12, 14, 16, 18, 20, 22 (Phases 6-12)")
print("  4. Compare via : !python -m scripts.compare_eval_results ...")

---

## Phase 6 — Comparaison in-distribution standardisée

Lit `final_validation_metrics.json` de chaque variante et produit un tableau métrique × variant. **Pas de rechargement des modèles**, post-hoc.

Métriques comparées :
- Pearson global + per-sample
- RMSE / MAE / Spread / Spread-RMSE ratio
- F1-p95 / F1-p99 (Pearson restreint aux centiles)
- RAPSD distance
- μ_HR ablation ratio (Δ_O3 architectural)

In [ ]:
# Phase 6 : comparaison in-distribution post-hoc
import json
import numpy as np

def load_metrics(d):
    """Load aligned baseline if present (post-round3 fix for apples-to-apples).

    Order de priorite :
    1. final_validation_metrics_aligned.json (formules nouvelles, baseline pre-recompute)
    2. final_validation_metrics.json (legacy, baseline V5-mini ou CorrDiff training-time)
    """
    p_aligned = Path(d) / "final_validation_metrics_aligned.json"
    if p_aligned.exists():
        return json.loads(p_aligned.read_text(encoding="utf-8"))
    p = Path(d) / "final_validation_metrics.json"
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))

m_v5 = load_metrics(ORACLE_DIR)   # baseline ou finetuned selon EVAL_VERSION
m_nc = load_metrics(NONCAUSAL_DIR)

if m_v5 is None:
    print(f"[ERREUR] Oracle metrics absent : {ORACLE_DIR}/final_validation_metrics.json")
if m_nc is None:
    print(f"[ERREUR] CorrDiff metrics absent : {NONCAUSAL_DIR}/final_validation_metrics.json")

if m_v5 and m_nc:
    def get(m, *path, default=None):
        v = m
        for k in path:
            if not isinstance(v, dict) or k not in v:
                return default
            v = v[k]
        return v

    rows = [
        ("Pearson global",     get(m_v5, "pearson_corr", "global"),         get(m_nc, "pearson_corr", "global"),         "haut"),
        ("Pearson per-sample", get(m_v5, "pearson_corr", "per_sample_avg"), get(m_nc, "pearson_corr", "per_sample_avg"), "haut"),
        ("RMSE",               get(m_v5, "rmse"),                            get(m_nc, "rmse"),                            "bas"),
        ("MAE",                get(m_v5, "mae"),                             get(m_nc, "mae"),                             "bas"),
        ("Spread (ens std)",   get(m_v5, "spread_mean"),                     get(m_nc, "spread_mean"),                     "calib"),
        ("F1-p95",             get(m_v5, "f1_extremes", "p95"),              get(m_nc, "f1_extremes", "p95"),              "haut"),
        ("F1-p99",             get(m_v5, "f1_extremes", "p99"),              get(m_nc, "f1_extremes", "p99"),              "haut"),
        ("RAPSD distance",     get(m_v5, "rapsd_distance"),                  get(m_nc, "rapsd_distance"),                  "bas"),
        ("mu_HR ablation",     get(m_v5, "mu_HR_ablation", "delta_signal_ratio_avg"), get(m_nc, "mu_HR_ablation", "delta_signal_ratio_avg"), "haut"),
    ]

    sr_v5 = (rows[4][1] or 0) / (rows[2][1] or 1)
    sr_nc = (rows[4][2] or 0) / (rows[2][2] or 1)
    rows.insert(5, ("Spread/RMSE", sr_v5, sr_nc, "vers 1"))

    print(f"\n{'Metrique':<22} {'Oracle':>12} {'CorrDiff':>12} {'D abs':>10} {'D rel %':>10} {'sens':>8} {'gagnant':>10}")
    print("-" * 100)
    summary = {}
    for name, v5, nc, sens in rows:
        if v5 is None or nc is None:
            print(f"{name:<22} {'n/a':>12} {'n/a':>12}")
            continue
        d_abs = v5 - nc
        d_rel = 100 * d_abs / nc if abs(nc) > 1e-12 else float('nan')
        if abs(d_rel) < 1.0:
            winner = "egal"
        elif sens == "haut" and d_abs > 0:
            winner = "Oracle"
        elif sens == "bas" and d_abs < 0:
            winner = "Oracle"
        elif sens == "calib":
            winner = "Oracle" if d_abs > 0 else "CorrDiff"
        elif sens == "vers 1":
            winner = "Oracle" if abs(v5 - 1) < abs(nc - 1) else "CorrDiff"
        else:
            winner = "CorrDiff"
        summary[name] = {"oracle": v5, "corrdiff": nc, "delta_abs": d_abs, "delta_rel_pct": d_rel, "winner": winner}
        print(f"{name:<22} {v5:>12.4f} {nc:>12.4f} {d_abs:>+10.4f} {d_rel:>+10.2f} {sens:>8} {winner:>10}")

    out = RESULTS_DIR / "phase6_in_distribution.json"
    out.write_text(json.dumps({
        "oracle_dir": str(ORACLE_DIR),
        "corrdiff_dir": str(NONCAUSAL_DIR),
        "comparison": summary,
        "raw_oracle_metrics": m_v5,
        "raw_corrdiff_metrics": m_nc,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n[OK] Phase 6 sauvegardee : {out}")

---

## Phase 7 — OOD réel sur EC-Earth3 + NorESM2-MM (4 runs alignés cGAN)

**Test inter-GCM** : on évalue Oracle et CorrDiff — entraînés sur ACCESS-CM2 — sur deux autres GCM (EC-Earth3 et NorESM2-MM) jamais vus à l'entraînement.

**Protocole** : pour chaque (variant × GCM), on utilise `st_cdgm.evaluation.aligned_eval.run_aligned_eval` (vendorisé depuis le cGAN de Rampal). Produit des fichiers `aligned_metrics_<GCM>_<variant>.json` contenant :
- **CDD** (Consecutive Dry Days) — fréquence des séquences sèches
- **Rx1Day** — précipitation maximale quotidienne
- **R10** — nombre de jours > 10 mm/j
- **Saisonnier** (DJF/JJA) — moyennes saisonnières
- **PSD** — Power Spectral Density (structure spectrale)
- + Pearson, RMSE, MAE comme références secondaires

**4 runs au total** : Oracle×EC-Earth3, Oracle×NorESM2-MM, CorrDiff×EC-Earth3, CorrDiff×NorESM2-MM. Plus l'in-dist ACCESS pour les deux pour la baseline.

**Δ_OOD** = `(Pearson_in - Pearson_ood) / Pearson_in` pour chaque (variant, GCM).

Coût estimé : ~3-5 min par run × 4 runs ≈ **15-25 min** (selon K_SAMPLES et longueur du sample subset).

In [ ]:
# ============================================================
# Phase 7 : runs OOD reels avec run_aligned_eval (vendored Rampal)
# + sidecar probabilistic_metrics : CRPS, RMSE, spread, CRPS-SS, rank histogram
# Sortie 1 : aligned_metrics_<GCM>_<variant>.json   (indices climatiques)
# Sortie 2 : probabilistic_metrics_<GCM>_<variant>.json (ensemble-based)
# ============================================================
import json
import time
import torch
import numpy as np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval

# Parametres
N_TIMES_OOD = 365
K_SAMPLES_OOD = 12          # bump 4 -> 12 pour CRPS empirique non bruite
N_STEPS_DIFF = 18

# --- Metriques probabilistes ---------------------------------------------

def _crps_empirical_fast(samples, obs):
    """CRPS empirique vectorise via tri (O(K log K) par point).

    samples : (K, ...) array, ensemble
    obs     : (...,) array, observation
    return  : (...,) array, CRPS par point
    Formule : E|X - y| - 0.5 * E|X - X'|, avec
              0.5 * (1/K^2) * sum_ij |xi - xj| = (1/K^2) * sum_k (2k - K - 1) * x_(k)
    """
    K = samples.shape[0]
    term1 = np.nanmean(np.abs(samples - obs[None]), axis=0)
    s = np.sort(samples, axis=0)
    k_idx = np.arange(1, K + 1).reshape((K,) + (1,) * (s.ndim - 1)).astype(np.float64)
    weights = 2.0 * k_idx - K - 1.0
    term2 = np.sum(weights * s, axis=0) / (K * K)
    return term1 - term2


def _crps_clim_per_pixel(truth):
    """CRPS de la climato empirique (distribution par pixel sur l'axe temps).

    Pour X, X' iid ~ distribution-truth(h,w) et y ~ idem :
        CRPS_clim(h,w) = E|X - y| - 0.5 * E|X - X'| = 0.5 * E|X - X'|
    (car E|X-Y| = E|X-X'| pour des copies iid).
    """
    T = truth.shape[0]
    t_sorted = np.sort(truth, axis=0)
    k_idx = np.arange(1, T + 1).reshape((T, 1, 1)).astype(np.float64)
    weights = 2.0 * k_idx - T - 1.0
    return np.sum(weights * t_sorted, axis=0) / (T * T)


def _rank_histogram(samples, truth):
    """Histogramme de Talagrand : rang de truth parmi les K samples (K+1 bins)."""
    K = samples.shape[0]
    rank = (samples < truth[None]).sum(axis=0).astype(np.int64)
    hist, _ = np.histogram(rank.flatten(), bins=np.arange(K + 2) - 0.5)
    return hist.astype(int).tolist()


def probabilistic_metrics(ens_log1p, truth_log1p):
    """Calcule toutes les metriques probabilistes apres conversion log1p -> mm/jour.

    ens_log1p   : (K, T, H, W) ensemble en espace log1p
    truth_log1p : (T, H, W) verite en espace log1p
    """
    ens = np.expm1(np.clip(ens_log1p.astype(np.float64), 0.0, None))
    truth = np.expm1(np.clip(truth_log1p.astype(np.float64), 0.0, None))

    pred_mean = ens.mean(axis=0)                         # (T, H, W)
    err2 = (pred_mean - truth) ** 2
    rmse_global = float(np.sqrt(np.nanmean(err2)))
    rmse_map_t = np.sqrt(np.nanmean(err2, axis=0))       # (H, W)

    ens_var = ens.var(axis=0)                            # (T, H, W)
    spread_global = float(np.sqrt(np.nanmean(ens_var)))
    spread_skill_ratio = float(spread_global / max(rmse_global, 1e-9))

    crps_model = _crps_empirical_fast(ens, truth)        # (T, H, W)
    crps_model_global = float(np.nanmean(crps_model))

    crps_clim_map = _crps_clim_per_pixel(truth)          # (H, W)
    crps_clim_global = float(np.nanmean(crps_clim_map))

    crps_ss = 1.0 - crps_model_global / max(crps_clim_global, 1e-9)

    hist = _rank_histogram(ens, truth)
    K = int(ens.shape[0])
    expected_per_bin = float(truth.size / (K + 1))
    chi2_uniform = float(sum((c - expected_per_bin) ** 2 / expected_per_bin for c in hist))

    return {
        "K_samples": K,
        "n_times": int(ens.shape[1]),
        "grid": [int(truth.shape[-2]), int(truth.shape[-1])],
        "rmse_global_mm": rmse_global,
        "rmse_map_mean_mm": float(np.nanmean(rmse_map_t)),
        "rmse_map_max_mm": float(np.nanmax(rmse_map_t)),
        "spread_global_mm": spread_global,
        "spread_skill_ratio": spread_skill_ratio,
        "crps_model_global_mm": crps_model_global,
        "crps_clim_global_mm": crps_clim_global,
        "crps_skill_score": float(crps_ss),
        "rank_histogram": hist,
        "rank_histogram_bins": list(range(len(hist))),
        "rank_histogram_chi2_vs_uniform": chi2_uniform,
        "_caveat": (
            "CRPS_clim computed from test-truth empirical distribution per pixel "
            "(includes the day under evaluation; slight optimistic bias for T~365). "
            "spread_skill_ratio ~1 = well-calibrated, <1 = under-dispersive, >1 = over-dispersive."
        ),
    }


# --- Collection ensemble complet ----------------------------------------

def collect_predictions_for_gcm(stack, gcm_tag, n_times=N_TIMES_OOD, K=K_SAMPLES_OOD):
    """Genere ensemble (K, T, H, W) + truth (T, H, W) + times."""
    lr_path, hr_path, in_dist = GCM_REGISTRY[gcm_tag]
    pipe = make_pipeline(lr_path, hr_path)
    ds = pipe.build_sequence_dataset(
        seq_len=int(CONFIG.data.seq_len), stride=1, as_torch=True,
    )

    ens_log, truths_log, times_list = [], [], []
    it = iter(ds)
    for i in range(n_times):
        try:
            sample = next(it)
        except StopIteration:
            break
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        ens = predict_with_stack(stack, batch, K=K, n_steps=N_STEPS_DIFF)  # (K, B=1, 1, H, W)
        # Reduit a (K, H, W) en droppant B et C
        while ens.dim() > 3:
            ens = ens.squeeze(1)
        ens_arr = ens.cpu().numpy()                      # (K, H, W)
        truth = (batch["baseline"][-1] + batch["residual"][-1]).cpu()
        truth_arr = truth.squeeze().numpy()              # (H, W)

        ens_log.append(ens_arr)
        truths_log.append(truth_arr)
        times_list.append(np.datetime64("1986-01-01") + np.timedelta64(i, "D"))

        if (i + 1) % 50 == 0:
            print(f"  [{gcm_tag} / {stack['variant']}] {i+1}/{n_times} pas evalues")

    ens_full = np.stack(ens_log, axis=1)                 # (K, T, H, W)
    truths = np.stack(truths_log, axis=0)                # (T, H, W)
    preds_mean = ens_full.mean(axis=0)                   # (T, H, W) pour run_aligned_eval
    times = np.array(times_list, dtype="datetime64[D]")
    return preds_mean, truths, times, ens_full


# --- Boucle principale --------------------------------------------------

print("=" * 70)
print(f"Phase 7 : 6 runs OOD ({N_TIMES_OOD} pas x K={K_SAMPLES_OOD} x {N_STEPS_DIFF} EDM steps)")
print(f"         + sidecar probabilistic_metrics par run")
print("=" * 70)
print()

all_results = {}      # aligned (Rampal)
all_prob = {}         # probabilistic (sidecar)
phase7_arrays = {}    # arrays in memory pour Phases 9-12

# Persistance : un dossier dedie sur Drive pour les arrays
ARRAYS_DIR = RESULTS_DIR / "phase7_runs"
ARRAYS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Resume directory : {ARRAYS_DIR}")


def _resume_from_disk(stack_name, gcm_tag, ckpt_dir):
    """Restaure un run depuis disque si npz + 2 JSONs existent et sont valides.

    Returns
    -------
    tuple (preds, truths, times, ens_full, aligned_data, prob_data) ou None.
    """
    arrays_path = ARRAYS_DIR / f"{stack_name}_{gcm_tag}.npz"
    aligned_path = ckpt_dir / f"aligned_metrics_{gcm_tag}_{stack_name.lower()}.json"
    prob_path = ckpt_dir / f"probabilistic_metrics_{gcm_tag}_{stack_name.lower()}.json"
    if not (arrays_path.exists() and aligned_path.exists() and prob_path.exists()):
        return None
    try:
        with np.load(arrays_path, allow_pickle=True) as z:
            preds = z["pred_mean"].astype(np.float32)
            truths = z["truth"].astype(np.float32)
            ens_full = z["ens"].astype(np.float32)      # cast back depuis float16
            times = z["times"]
        aligned_data = json.loads(aligned_path.read_text(encoding="utf-8"))
        prob_data = json.loads(prob_path.read_text(encoding="utf-8"))
        return preds, truths, times, ens_full, aligned_data, prob_data
    except Exception as e:
        print(f"  [WARN] {stack_name}_{gcm_tag} restore failed ({type(e).__name__}: {e}) - recompute")
        return None


def _persist_to_disk(stack_name, gcm_tag, preds, truths, ens_full, times):
    """Sauvegarde arrays npz sur Drive. ens en float16 pour economiser l espace."""
    arrays_path = ARRAYS_DIR / f"{stack_name}_{gcm_tag}.npz"
    try:
        np.savez_compressed(
            str(arrays_path),
            pred_mean=preds.astype(np.float32),
            truth=truths.astype(np.float32),
            ens=ens_full.astype(np.float16),    # half precision pour ens (gain ~50%)
            times=np.array(times, dtype="datetime64[D]"),
        )
        size_mb = arrays_path.stat().st_size / 1e6
        return True, size_mb
    except Exception as e:
        print(f"       [WARN] persist failed : {type(e).__name__}: {e}")
        return False, 0.0


t_global = time.time()

# BUG fix : on ecrit dans ORACLE_DIR (= ORACLE_FINETUNED_DIR si EVAL_VERSION='finetuned')
# pour ne pas polluer le baseline V5_DIR. CorrDiff JSONs dans RESULTS_DIR pour
# preserver ckpt_noncausal/ intact.
_corrdiff_ckpt_dir = RESULTS_DIR / "corrdiff_phase7"
_corrdiff_ckpt_dir.mkdir(parents=True, exist_ok=True)
for stack_name, stack, ckpt_dir in [("V5", stack_v5, ORACLE_DIR),
                                      ("Noncausal", stack_nc, _corrdiff_ckpt_dir)]:
    for gcm_tag in ["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]:
        run_label = f"{stack_name}_{gcm_tag}"
        t0 = time.time()
        print(f"\n=== RUN {run_label} ===")

        if not GCM_REGISTRY[gcm_tag][0].exists():
            print(f"  [SKIP] {GCM_REGISTRY[gcm_tag][0]} absent")
            continue

        # === RESUME : check si run deja persiste ===
        resumed = _resume_from_disk(stack_name, gcm_tag, ckpt_dir)
        if resumed is not None:
            preds, truths, times, ens_full, aligned_data, prob_data = resumed
            all_results[run_label] = aligned_data
            all_prob[run_label] = prob_data
            phase7_arrays[run_label] = {
                "pred_mean": preds, "truth": truths,
                "ens": ens_full, "times": times,
            }
            crps = prob_data.get('crps_model_global_mm', 0.0)
            crps_ss = prob_data.get('crps_skill_score', 0.0)
            rmse = prob_data.get('rmse_global_mm', 0.0)
            spread = prob_data.get('spread_skill_ratio', 0.0)
            print(f"  [RESUME] charge depuis disque (skip compute, gain ~90 min)")
            print(f"           CRPS={crps:.3f}mm  CRPS-SS={crps_ss:+.3f}  "
                  f"RMSE={rmse:.3f}mm  spread/skill={spread:.3f}")
            continue

        try:
            preds, truths, times, ens_full = collect_predictions_for_gcm(stack, gcm_tag)
        except Exception as e:
            print(f"  [ERREUR] {type(e).__name__}: {e}")
            continue

        if len(preds) == 0:
            print(f"  [SKIP] Pas de samples retournes")
            continue

        # 1. Vendored Rampal indices
        out_path = ckpt_dir / f"aligned_metrics_{gcm_tag}_{stack_name.lower()}.json"
        try:
            run_aligned_eval(
                pred_fields=preds, truth_fields=truths, times=times,
                out_path=str(out_path),
                gcm=gcm_tag, run_variant=stack_name.lower(),
                in_distribution=GCM_REGISTRY[gcm_tag][2],
                space="log1p", k_samples=K_SAMPLES_OOD,
            )
            with open(out_path, "r", encoding="utf-8") as f:
                all_results[run_label] = json.load(f)
        except Exception as e:
            print(f"  [ERREUR run_aligned_eval] {type(e).__name__}: {e}")

        # 2. Sidecar probabiliste
        prob_path = ckpt_dir / f"probabilistic_metrics_{gcm_tag}_{stack_name.lower()}.json"
        try:
            prob = probabilistic_metrics(ens_full, truths)
            prob.update({
                "gcm": gcm_tag,
                "run_variant": stack_name.lower(),
                "in_distribution": GCM_REGISTRY[gcm_tag][2],
            })
            prob_path.write_text(json.dumps(prob, ensure_ascii=False, indent=2),
                                  encoding="utf-8")
            all_prob[run_label] = prob
            phase7_arrays[run_label] = {
                "pred_mean": preds.astype("float32"),
                "truth": truths.astype("float32"),
                "ens": ens_full.astype("float32"),
                "times": times,
            }
            # Persiste sur Drive pour resume futur
            ok, size_mb = _persist_to_disk(stack_name, gcm_tag, preds, truths, ens_full, times)
            if ok:
                print(f"       [PERSIST] {stack_name}_{gcm_tag}.npz sauve ({size_mb:.0f} MB sur Drive)")
            print(f"  [OK] aligned + probabilistic ({(time.time()-t0):.1f}s)")
            print(f"       CRPS={prob['crps_model_global_mm']:.3f}mm  "
                  f"CRPS-SS={prob['crps_skill_score']:+.3f}  "
                  f"RMSE={prob['rmse_global_mm']:.3f}mm  "
                  f"spread/skill={prob['spread_skill_ratio']:.3f}")
        except Exception as e:
            print(f"  [ERREUR probabilistic_metrics] {type(e).__name__}: {e}")

print()
print(f"[OK] Phase 7 terminee en {(time.time()-t_global)/60:.1f} min")

# --- Recapitulatif ------------------------------------------------------

print()
print("=" * 70)
print("RECAPITULATIF Delta_OOD (indices Rampal + probabiliste)")
print("=" * 70)
for variant in ["V5", "Noncausal"]:
    _disp = "CorrDiff" if variant == "Noncausal" else "ORACLE"
    print(f"\n--- {_disp} ---")
    in_dist = all_results.get(f"{variant}_ACCESS-CM2", {})
    in_prob = all_prob.get(f"{variant}_ACCESS-CM2", {})
    if in_dist:
        psd_in = in_dist.get("psd_distance")
        crps_in = in_prob.get("crps_model_global_mm")
        crpsss_in = in_prob.get("crps_skill_score")
        if psd_in is not None:
            print(f"  ID  ACCESS-CM2  PSD={psd_in:.4f}  CRPS={crps_in:.3f}mm  CRPS-SS={crpsss_in:+.3f}")
    for ood in ["EC-Earth3", "NorESM2-MM"]:
        r = all_results.get(f"{variant}_{ood}", {})
        p = all_prob.get(f"{variant}_{ood}", {})
        if r and p:
            print(f"  OOD {ood:<12s} PSD={r.get('psd_distance'):.4f}  "
                  f"CRPS={p.get('crps_model_global_mm'):.3f}mm  "
                  f"CRPS-SS={p.get('crps_skill_score'):+.3f}  "
                  f"spread/skill={p.get('spread_skill_ratio'):.3f}")

# --- Sauvegarde recap ---------------------------------------------------

recap_path = RESULTS_DIR / "phase7_ood_aligned.json"
recap_path.write_text(json.dumps({
    "n_times": N_TIMES_OOD, "K_samples": K_SAMPLES_OOD, "n_steps": N_STEPS_DIFF,
    "runs": all_results,
    "probabilistic": all_prob,
}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"\n[OK] Recap sauvegarde : {recap_path}")

---

## Phase 8 — Interprétabilité visuelle de la causalité

Cette phase **génère un maximum de visualisations** pour démontrer l'apport interprétable du chemin causal d'Oracle vs le baseline CorrDiff. Toutes les figures sont sauvegardées en PNG dans `RESULTS_DIR/phase8_figures/`.

### 8 visualisations produites

1. **DAG appris Oracle** — heatmap de la matrice `A_dag` 6×6 (qui cause quoi)
2. **Distribution α (gate skip-connection)** — histogramme de la fraction du chemin causal
3. **Interventions `do(·)` — cartes Δ_pred** — 3 interventions × 2 variants = 6 cartes côte à côte
4. **Sensibilité par variable LR** — gradient |∂pr_HR/∂var_LR| par variable (15 panneaux)
5. **Ablation A_dag** — μ_HR avec A_dag réel vs A_dag=0 (cartes side-by-side)
6. **Comparaison spatiale Oracle vs CorrDiff** — same input, deux sorties côte à côte + différence
7. **Spectre radial (RAPSD)** — courbes Oracle vs CorrDiff vs vérité
8. **Histogramme intensités** — distribution des prédictions vs vérité (queue lourde)

### Tableaux

- Q_int par modèle (Oracle, CorrDiff) sur 3 interventions standardisées
- Magnitude et signe Δ_pred par intervention

Coût estimé : ~5-8 min sur A100 (peu de samples, beaucoup de figures).

In [ ]:
# ============================================================
# Phase 8 : Interpretabilite visuelle
# Genere 8 figures PNG + tableaux Q_int + sauvegarde JSON.
# ============================================================
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm
from pathlib import Path

FIG_DIR = RESULTS_DIR / "phase8_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures -> {FIG_DIR}")

# === 1. DAG appris V5 (heatmap A_dag) =======================================
if stack_v5["A_dag"] is not None:
    A = stack_v5["A_dag"].numpy()
    # Mask diagonale (no self-loop par convention)
    A_disp = A.copy()
    np.fill_diagonal(A_disp, 0)

    fig, ax = plt.subplots(1, 1, figsize=(7, 6))
    im = ax.imshow(A_disp, cmap="RdBu_r", vmin=-np.abs(A_disp).max(), vmax=np.abs(A_disp).max())
    ax.set_title("DAG appris (ORACLE) — matrice d'adjacence A_dag\n(diagonale masquee)", fontsize=11)
    # Labels : par convention, lignes = sources, colonnes = cibles
    var_labels = ["GP850_spat", "GP850→GP500", "GP500_spat", "GP500→GP250", "GP250_spat", "SP_HR"]
    ax.set_xticks(range(len(var_labels))); ax.set_xticklabels(var_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(var_labels))); ax.set_yticklabels(var_labels)
    ax.set_xlabel("Cible (effet)")
    ax.set_ylabel("Source (cause)")
    for i in range(A_disp.shape[0]):
        for j in range(A_disp.shape[1]):
            if i != j:
                ax.text(j, i, f"{A_disp[i,j]:.2f}", ha="center", va="center",
                        color="white" if abs(A_disp[i,j]) > 0.3 else "black", fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "01_dag_oracle.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("[OK] Figure 1 : DAG ORACLE")
else:
    print("[SKIP] Figure 1 : A_dag indisponible")

# === 2. Distribution alpha (gate skip-connection) ==========================
if stack_v5["skip_block"] is not None:
    alphas = []
    it = iter(test_dataset)
    skip = stack_v5["skip_block"]
    rh = stack_v5["regression_head"]
    enc = stack_v5["encoder"]
    rcn = stack_v5["rcn_runner"]
    with torch.no_grad():
        for i in range(min(32, 100)):
            try:
                s = next(it)
            except StopIteration:
                break
            b = convert_sample_to_batch(s, builder, DEVICE)
            lr = b["lr"].to(DEVICE)
            H_init = enc.init_state(b["hetero"]).to(DEVICE)
            drivers = [lr[t] for t in range(lr.shape[0])]
            seq = rcn.run(H_init, drivers, reconstruction_sources=None)
            mu_c = rh(seq.states[-1])
            tshape = b["residual"][-1].to(DEVICE).shape
            if tshape[-2:] != mu_c.shape[-2:]:
                mu_c = torch.nn.functional.interpolate(mu_c, size=tshape[-2:], mode="bilinear", align_corners=False)
            lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
            _, a = skip(lr_last, mu_c)
            alphas.append(a.cpu().numpy().flatten())
    alphas = np.concatenate(alphas) if alphas else np.array([])
    if alphas.size > 0:
        fig, ax = plt.subplots(1, 1, figsize=(8, 4))
        ax.hist(alphas, bins=30, color="steelblue", edgecolor="black", alpha=0.75)
        ax.axvline(0.6, color="red", linestyle="--", label="alpha_floor=0.6")
        ax.axvline(alphas.mean(), color="green", linestyle="-", label=f"moyenne={alphas.mean():.3f}")
        ax.set_xlabel("alpha (fraction chemin causal)")
        ax.set_ylabel("Frequence")
        ax.set_title(f"Distribution alpha (skip-connection gate) — n={len(alphas)} samples\n"
                      f"alpha > 0.6 sur {(alphas > 0.6).mean()*100:.1f}% des cas (preservation O3)")
        ax.legend()
        ax.set_xlim(0, 1)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "02_alpha_distribution.png", dpi=120, bbox_inches="tight")
        plt.close()
        print(f"[OK] Figure 2 : alpha mean={alphas.mean():.3f}, prop>0.6={((alphas>0.6).mean()*100):.1f}%")
else:
    print("[SKIP] Figure 2 : skip_block absent")

# === 3. Interventions do(.) - protocole multi-sample en unites physiques ====
from scripts.intervention_test import INTERVENTIONS, resolve_variable_indices, apply_intervention
lr_vars = list(CONFIG.data.lr_variables)
resolved = resolve_variable_indices(lr_vars)

# --- Build standardization dict (per-variable mean/std, spatial average) ---
# Sans ce dict, le multiplicateur agit sur des z-scores -> mean delta ~0,
# signe aleatoire (Pawlowski 2020, Janzing & Mejia 2024 https://arxiv.org/abs/2406.11601).
standardization = None
try:
    pipe_stats = make_pipeline(GCM_REGISTRY["ACCESS-CM2"][0], GCM_REGISTRY["ACCESS-CM2"][1])
    raw_stats = pipe_stats.get_lr_stats()
    standardization = {}
    for v in lr_vars:
        try:
            mu = float(raw_stats["mean"][v].mean().values)
            sd = float(raw_stats["std"][v].mean().values)
            standardization[v] = {"mean": mu, "std": max(sd, 1e-12)}
        except Exception as e:
            print(f"  [WARN] stats {v}: {e}")
    print(f"[OK] standardization dict : {len(standardization)}/{len(lr_vars)} variables")
except Exception as e:
    print(f"[WARN] standardization indisponible ({e}) - signes peu fiables")

N_INT_SAMPLES = 30
K_INT = 2
print(f"Protocole Q_int : N={N_INT_SAMPLES} samples x K={K_INT} ensemble")
print(f"Espace : {'PHYSIQUE (correct)' if standardization else 'z-SCORE (signe aleatoire)'}")

n_int = sum(1 for s in resolved if s["variable_idx"] is not None)
intervention_results = {"V5": [], "Noncausal": []}
per_sample_deltas = {"V5": {}, "Noncausal": {}}

if n_int > 0:
    # --- Boucle multi-sample ---
    for spec in resolved:
        if spec["variable_idx"] is None:
            continue
        deltas_v5, deltas_nc = [], []
        it_int = iter(test_dataset)
        for k_sample in range(N_INT_SAMPLES):
            try:
                s_k = next(it_int)
            except StopIteration:
                break
            b_k = convert_sample_to_batch(s_k, builder, DEVICE)
            b_int = dict(b_k)
            b_int["lr"] = apply_intervention(b_k["lr"], spec, standardization=standardization)
            with torch.no_grad():
                pn_v5 = predict_with_stack(stack_v5, b_k, K=K_INT, n_steps=18).nanmean(0).squeeze().cpu().numpy()
                pi_v5 = predict_with_stack(stack_v5, b_int, K=K_INT, n_steps=18).nanmean(0).squeeze().cpu().numpy()
                pn_nc = predict_with_stack(stack_nc, b_k, K=K_INT, n_steps=18).nanmean(0).squeeze().cpu().numpy()
                pi_nc = predict_with_stack(stack_nc, b_int, K=K_INT, n_steps=18).nanmean(0).squeeze().cpu().numpy()
            deltas_v5.append(float(np.nanmean(pi_v5 - pn_v5)))
            deltas_nc.append(float(np.nanmean(pi_nc - pn_nc)))
        per_sample_deltas["V5"][spec["name"]] = deltas_v5
        per_sample_deltas["Noncausal"][spec["name"]] = deltas_nc
        print(f"  [{spec['name']:30s}] N={len(deltas_v5)} samples")

    # --- Bootstrap CI + sign test ---
    def _boot_ci(vals, n_boot=1000, alpha=0.05):
        arr = np.array(vals, dtype=np.float64)
        if arr.size == 0: return None, None, None, None
        rng = np.random.default_rng(42)
        boots = np.array([rng.choice(arr, arr.size, replace=True).mean()
                          for _ in range(n_boot)])
        lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
        return float(arr.mean()), float(lo), float(hi), float((arr > 0).mean())

    for spec in resolved:
        if spec["variable_idx"] is None: continue
        for variant in ["V5", "Noncausal"]:
            d = per_sample_deltas[variant][spec["name"]]
            mean, lo, hi, fp = _boot_ci(d)
            sign_pred = int(np.sign(mean)) if mean is not None and mean != 0 else 0
            sign_exp = int(spec["expected_sign"])
            match = int(sign_pred == sign_exp)
            ci_excl_zero = int((lo > 0 or hi < 0)) if lo is not None else 0
            intervention_results[variant].append({
                "intervention": spec["name"], "variable": spec["variable_name"],
                "delta_mean": mean, "delta_ci_lo": lo, "delta_ci_hi": hi,
                "fraction_positive": fp, "ci_excludes_zero": ci_excl_zero,
                "sign_pred": sign_pred, "sign_expected": sign_exp, "match": match,
                "n_samples": len(d),
            })

    # --- Print summary ---
    print()
    print("=" * 72)
    print(f"Q_int (sign test multi-sample, N={N_INT_SAMPLES}, bootstrap CI 95%)")
    print("=" * 72)
    for variant in ["V5", "Noncausal"]:
        results = intervention_results[variant]
        q_int = float(np.mean([r["match"] for r in results])) if results else 0.0
        n_match = sum(r["match"] for r in results)
        _disp = "CorrDiff" if variant == "Noncausal" else "ORACLE"
        print(f"\n{_disp:10s} : Q_int = {q_int:.3f}  ({n_match}/{len(results)} signes corrects)")
        for r in results:
            ci_tag = "**" if r["ci_excludes_zero"] else "  "
            sign_tag = "OK" if r["match"] else "KO"
            dm = r["delta_mean"] if r["delta_mean"] is not None else float("nan")
            lo_, hi_ = r["delta_ci_lo"], r["delta_ci_hi"]
            print(f"  {ci_tag} {r['intervention']:28s} delta={dm:+.5f} "
                  f"CI[{lo_:+.5f},{hi_:+.5f}] "
                  f"frac+={r['fraction_positive']*100:3.0f}%  [{sign_tag}]")
    print()
    print("Legende : ** = CI exclut zero (effet significatif)")
    print("          delta_t850 regime-dependant pour NZ (Gibson 2024, NIWA proj.)")

    # --- Figure : single-sample pour visualisation ---
    fig, axes = plt.subplots(n_int, 4, figsize=(16, 3.5 * n_int))
    if n_int == 1:
        axes = axes.reshape(1, -1)
    sample_first = next(iter(test_dataset))
    batch_first = convert_sample_to_batch(sample_first, builder, DEVICE)
    row_idx = 0
    for spec in resolved:
        if spec["variable_idx"] is None: continue
        with torch.no_grad():
            pred_norm_v5 = predict_with_stack(stack_v5, batch_first, K=4, n_steps=18).nanmean(0).squeeze().numpy()
            pred_norm_nc = predict_with_stack(stack_nc, batch_first, K=4, n_steps=18).nanmean(0).squeeze().numpy()
        batch_int = dict(batch_first)
        batch_int["lr"] = apply_intervention(batch_first["lr"], spec, standardization=standardization)
        with torch.no_grad():
            pred_int_v5 = predict_with_stack(stack_v5, batch_int, K=4, n_steps=18).nanmean(0).squeeze().numpy()
            pred_int_nc = predict_with_stack(stack_nc, batch_int, K=4, n_steps=18).nanmean(0).squeeze().numpy()
        delta_v5 = pred_int_v5 - pred_norm_v5
        delta_nc = pred_int_nc - pred_norm_nc
        vmax = max(abs(delta_v5).max(), abs(delta_nc).max(), 0.01)
        axes[row_idx, 0].imshow(pred_norm_v5, cmap="viridis")
        axes[row_idx, 0].set_title(f"ORACLE normal", fontsize=10)
        axes[row_idx, 1].imshow(pred_int_v5, cmap="viridis")
        axes[row_idx, 1].set_title(f"ORACLE + {spec['name']}", fontsize=10)
        im_v5 = axes[row_idx, 2].imshow(delta_v5, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axes[row_idx, 2].set_title(f"Delta ORACLE (sample 0)", fontsize=10)
        plt.colorbar(im_v5, ax=axes[row_idx, 2], shrink=0.7)
        im_nc = axes[row_idx, 3].imshow(delta_nc, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axes[row_idx, 3].set_title(f"Delta CorrDiff (sample 0)", fontsize=10)
        plt.colorbar(im_nc, ax=axes[row_idx, 3], shrink=0.7)
        for a in axes[row_idx, :]:
            a.set_xticks([]); a.set_yticks([])
        row_idx += 1
    plt.suptitle("Phase 8 : Cartes Delta_pred (sample 0)\n"
                  "Q_int statistique calcule sur N=30 samples (cf. resume au-dessus)",
                  fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "03_interventions_maps.png", dpi=120, bbox_inches="tight")
    plt.close()
    print(f"\n[OK] Figure 3 : {n_int} interventions x 4 panneaux (visualisation sample 0)")

    # Tableau Q_int final
    q_v5 = float(np.mean([r["match"] for r in intervention_results["V5"]])) if intervention_results["V5"] else 0.0
    q_nc = float(np.mean([r["match"] for r in intervention_results["Noncausal"]])) if intervention_results["Noncausal"] else 0.0
    print()
    print(f"Q_int Oracle     : {q_v5:.3f}  ({sum(r['match'] for r in intervention_results['V5'])}/{len(intervention_results['V5'])} signes corrects)")
    print(f"Q_int CorrDiff   : {q_nc:.3f}  ({sum(r['match'] for r in intervention_results['Noncausal'])}/{len(intervention_results['Noncausal'])} signes corrects)")

# === 4. Sensibilite par variable LR (gradients) ==============================
print()
print("Computation de la sensibilite par variable...")
sensitivities = {"V5": {}, "Noncausal": {}}
sample_sens = next(iter(test_dataset))
batch_sens = convert_sample_to_batch(sample_sens, builder, DEVICE)

for stack_name, stack in [("V5", stack_v5), ("Noncausal", stack_nc)]:
    enc, rcn, rh, skip = stack["encoder"], stack["rcn_runner"], stack["regression_head"], stack["skip_block"]
    lr = batch_sens["lr"].clone().to(DEVICE).requires_grad_(True)
    H_init = enc.init_state(batch_sens["hetero"]).to(DEVICE)
    drivers = [lr[t] for t in range(lr.shape[0])]
    seq = rcn.run(H_init, drivers, reconstruction_sources=None)
    mu_c = rh(seq.states[-1])
    tshape = batch_sens["residual"][-1].to(DEVICE).shape
    if tshape[-2:] != mu_c.shape[-2:]:
        mu_c = torch.nn.functional.interpolate(mu_c, size=tshape[-2:], mode="bilinear", align_corners=False)
    if skip is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu, _ = skip(lr_last, mu_c)
    else:
        mu = mu_c
    target_scalar = mu.abs().sum()
    target_scalar.backward()
    # Gradient absolu moyenne dans le temps et dans l'espace LR
    # Reduit toutes les dims sauf la derniere (canal C) pour obtenir [C_lr]
    # Supporte (T, N, C) graphe, (T, C, H, W) grille, (B, T, C, H, W) batch
    g = lr.grad.detach().abs()
    if g.dim() == 3:        # (T, N, C) graphe : channels en dernier
        grad = g.mean(dim=(0, 1))
    elif g.dim() == 4:      # (T, C, H, W) grille
        grad = g.mean(dim=(0, 2, 3))
    elif g.dim() == 5:      # (B, T, C, H, W)
        grad = g.mean(dim=(0, 1, 3, 4))
    else:
        grad = g.reshape(-1, g.shape[-1]).mean(dim=0)
    for i, v in enumerate(lr_vars):
        sensitivities[stack_name][v] = float(grad[i].cpu())

# Plot par variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(lr_vars))
w = 0.35
v5_vals = [sensitivities["V5"][v] for v in lr_vars]
nc_vals = [sensitivities["Noncausal"][v] for v in lr_vars]
axes[0].bar(x - w/2, v5_vals, w, label="ORACLE", color="steelblue")
axes[0].bar(x + w/2, nc_vals, w, label="CorrDiff", color="orange")
axes[0].set_xticks(x); axes[0].set_xticklabels(lr_vars, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("|d mu_HR / d var_LR| moyenne")
axes[0].set_title("Sensibilite de mu_HR par variable LR (ORACLE vs CorrDiff)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Ratio V5/NC pour voir ou la causalite change la sensibilite
ratios = [v5_vals[i] / max(nc_vals[i], 1e-12) for i in range(len(lr_vars))]
axes[1].bar(x, ratios, color="purple", alpha=0.7)
axes[1].axhline(1.0, color="red", linestyle="--", label="ratio=1 (egal)")
axes[1].set_xticks(x); axes[1].set_xticklabels(lr_vars, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Ratio sensibilite ORACLE / CorrDiff")
axes[1].set_title("Ratio des sensibilites ORACLE / CorrDiff — > 1 = ORACLE utilise plus cette variable")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_sensitivity_per_variable.png", dpi=120, bbox_inches="tight")
plt.close()
print("[OK] Figure 4 : sensibilites par variable")

# === 5. Ablation A_dag (mu_HR avec vs sans DAG) ============================
if stack_v5["A_dag"] is not None:
    print()
    print("Ablation A_dag sur ORACLE...")
    rcn_cell = stack_v5["rcn_runner"].cell
    A_orig = rcn_cell.A_dag.detach().clone()
    sample_abl = next(iter(test_dataset))
    batch_abl = convert_sample_to_batch(sample_abl, builder, DEVICE)

    with torch.no_grad():
        # mu_HR avec A_dag normal
        mu_full = predict_with_stack(stack_v5, batch_abl, K=1, n_steps=18).nanmean(0).squeeze().numpy()
        # Ablation : A_dag := 0
        rcn_cell.A_dag.data.zero_()
        mu_ablated = predict_with_stack(stack_v5, batch_abl, K=1, n_steps=18).nanmean(0).squeeze().numpy()
        # Restaurer
        rcn_cell.A_dag.data.copy_(A_orig)

    delta_ablation = mu_full - mu_ablated
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].imshow(mu_full, cmap="viridis")
    axes[0].set_title("ORACLE avec A_dag appris")
    axes[1].imshow(mu_ablated, cmap="viridis")
    axes[1].set_title("ORACLE avec A_dag = 0 (ablation)")
    vmax = max(abs(delta_ablation).max(), 1e-3)
    im = axes[2].imshow(delta_ablation, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    delta_signal_ratio = float(np.abs(delta_ablation).mean() / max(np.abs(mu_full).mean(), 1e-12))
    axes[2].set_title(f"Delta (Delta/signal = {delta_signal_ratio:.2%})")
    plt.colorbar(im, ax=axes[2], shrink=0.7)
    for a in axes:
        a.set_xticks([]); a.set_yticks([])
    plt.suptitle("Ablation A_dag sur ORACLE — la difference quantifie l'apport du DAG", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "05_ablation_A_dag.png", dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[OK] Figure 5 : Delta/signal = {delta_signal_ratio:.2%}")

# === 6. Comparaison spatiale V5 vs Noncausal (meme input) ==================
sample_comp = next(iter(test_dataset))
batch_comp = convert_sample_to_batch(sample_comp, builder, DEVICE)
with torch.no_grad():
    pred_v5 = predict_with_stack(stack_v5, batch_comp, K=4, n_steps=18).nanmean(0).squeeze().numpy()
    pred_nc = predict_with_stack(stack_nc, batch_comp, K=4, n_steps=18).nanmean(0).squeeze().numpy()
truth = (batch_comp["baseline"][-1] + batch_comp["residual"][-1]).cpu().squeeze().numpy()
diff = pred_v5 - pred_nc

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
vmin, vmax = np.nanmin([pred_v5, pred_nc, truth]), np.nanmax([pred_v5, pred_nc, truth])
axes[0,0].imshow(truth, cmap="viridis", vmin=vmin, vmax=vmax)
axes[0,0].set_title("Verite terrain (HR cible)")
axes[0,1].imshow(pred_v5, cmap="viridis", vmin=vmin, vmax=vmax)
axes[0,1].set_title("ORACLE prediction")
axes[1,0].imshow(pred_nc, cmap="viridis", vmin=vmin, vmax=vmax)
axes[1,0].set_title("CorrDiff prediction")
diff_max = max(abs(diff).max(), 1e-3)
im = axes[1,1].imshow(diff, cmap="RdBu_r", vmin=-diff_max, vmax=diff_max)
axes[1,1].set_title(f"ORACLE - CorrDiff\n(mean abs = {np.abs(diff).mean():.4f})")
plt.colorbar(im, ax=axes[1,1], shrink=0.7)
for a in axes.flatten():
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("Comparaison spatiale ORACLE vs CorrDiff (meme input)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_spatial_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("[OK] Figure 6 : comparaison spatiale")

# === 7. Spectre radial RAPSD ================================================
def radial_power_spectrum(field):
    """RAPSD 1D : moyenne radiale de la PSD 2D."""
    f = np.fft.fft2(field)
    psd2d = np.abs(f) ** 2
    H, W = field.shape
    cy, cx = H // 2, W // 2
    Y, X = np.indices(field.shape)
    R = np.sqrt((Y - cy) ** 2 + (X - cx) ** 2).astype(int)
    R_max = min(cx, cy)
    radial_mean = np.zeros(R_max)
    for r in range(R_max):
        mask = R == r
        if mask.sum() > 0:
            radial_mean[r] = psd2d[mask].mean()
    return radial_mean

psd_v5 = radial_power_spectrum(pred_v5)
psd_nc = radial_power_spectrum(pred_nc)
psd_truth = radial_power_spectrum(truth)
k = np.arange(1, len(psd_v5) + 1)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.loglog(k, psd_truth[:len(k)], label="Verite", color="black", linewidth=2)
ax.loglog(k, psd_v5[:len(k)], label="ORACLE", color="steelblue", linewidth=1.5)
ax.loglog(k, psd_nc[:len(k)], label="CorrDiff", color="orange", linewidth=1.5)
ax.set_xlabel("Nombre d'onde radial k")
ax.set_ylabel("PSD radiale")
ax.set_title("Spectre radial (RAPSD) — fidelite des structures spatiales par echelle")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.savefig(FIG_DIR / "07_rapsd_comparison.png", dpi=120, bbox_inches="tight")
plt.close()
print("[OK] Figure 7 : RAPSD")

# === 8. Histogramme intensites ==============================================
fig, ax = plt.subplots(1, 1, figsize=(9, 5))
bins = np.linspace(0, max(np.nanmax(truth), np.nanmax(pred_v5), np.nanmax(pred_nc)), 60)
ax.hist(truth.flatten(), bins=bins, alpha=0.6, label="Verite", color="black", density=True)
ax.hist(pred_v5.flatten(), bins=bins, alpha=0.5, label="ORACLE", color="steelblue", density=True)
ax.hist(pred_nc.flatten(), bins=bins, alpha=0.5, label="CorrDiff", color="orange", density=True)
ax.set_yscale("log")
ax.set_xlabel("Intensite (log1p mm/jour)")
ax.set_ylabel("Densite (log)")
ax.set_title("Distribution des intensites (queue lourde des extremes)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "08_intensity_histogram.png", dpi=120, bbox_inches="tight")
plt.close()
print("[OK] Figure 8 : histogramme intensites")

# === Sauvegarde JSON de tous les resultats =================================
results_full = {
    "interventions": intervention_results,
    "Q_int": {
        "V5": float(np.mean([r["match"] for r in intervention_results["V5"]])) if intervention_results["V5"] else None,
        "Noncausal": float(np.mean([r["match"] for r in intervention_results["Noncausal"]])) if intervention_results["Noncausal"] else None,
    },
    "sensitivities": sensitivities,
    "ablation_A_dag_delta_signal_ratio": delta_signal_ratio if stack_v5["A_dag"] is not None else None,
    "alpha_distribution": {
        "mean": float(alphas.mean()) if stack_v5["skip_block"] is not None and len(alphas) > 0 else None,
        "std": float(alphas.std()) if stack_v5["skip_block"] is not None and len(alphas) > 0 else None,
        "prop_above_floor": float((alphas > 0.6).mean()) if stack_v5["skip_block"] is not None and len(alphas) > 0 else None,
    },
    "figures": sorted([str(p.name) for p in FIG_DIR.glob("*.png")]),
}
(RESULTS_DIR / "phase8_interpretability.json").write_text(
    json.dumps(results_full, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print()
print("=" * 70)
print("Phase 8 terminee — 8 figures + JSON resultats")
print("=" * 70)
print(f"Q_int ORACLE     : {results_full['Q_int']['V5']}")
print(f"Q_int CorrDiff   : {results_full['Q_int']['Noncausal']}")
print(f"Figures dans : {FIG_DIR}")
print(f"JSON : {RESULTS_DIR / 'phase8_interpretability.json'}")


---

## Phase 9 — Diagnostics climate-standard

Quatre figures attendues par la communaute climate downscaling (CorrDiff, Rampal cGAN, IPCC AR6 Ch. 11) :

1. **Q-Q plot tail** par (variant × GCM) sur jours pluvieux — fidelite distributionnelle dans la queue
2. **Return-period Gumbel** sur max domain-wide journaliers — extrapolation des extremes
3. **Reliability diagram** pour P(precip > {1, 10, 50} mm) — calibration probabiliste
4. **Fractions Skill Score (FSS)** vs taille de voisinage — echelle a laquelle le modele devient skillful

Lit `phase7_arrays` (rempli par Phase 7).


In [ ]:
# Phase 9 — Diagnostics climate-standard
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

PHASE9_DIR = RESULTS_DIR / "phase9_climate_standards"
PHASE9_DIR.mkdir(parents=True, exist_ok=True)

# Display names pour figures (data structures gardent "Noncausal")
DISPLAY = {"V5": "ORACLE", "Noncausal": "CorrDiff"}

if "phase7_arrays" not in dir() or not phase7_arrays:
    print("[ERREUR] phase7_arrays non disponible — relance Phase 7 d'abord.")
else:
    print(f"phase7_arrays : {len(phase7_arrays)} runs disponibles")

    # === 1. Q-Q plot tail par (variant, GCM) ===========================
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    quantiles = np.linspace(0.5, 0.9999, 200)
    for r, variant in enumerate(["V5", "Noncausal"]):
        for c, gcm in enumerate(["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]):
            ax = axes[r, c]
            run_label = f"{variant}_{gcm}"
            if run_label not in phase7_arrays:
                ax.text(0.5, 0.5, "(no data)", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(f"{DISPLAY[variant]} / {gcm}"); ax.axis("off"); continue
            d = phase7_arrays[run_label]
            pred_mm = np.expm1(np.clip(d["pred_mean"].astype(np.float64), 0, None))
            truth_mm = np.expm1(np.clip(d["truth"].astype(np.float64), 0, None))
            pred_wet = pred_mm[pred_mm > 1.0]; truth_wet = truth_mm[truth_mm > 1.0]
            if pred_wet.size == 0 or truth_wet.size == 0:
                ax.text(0.5, 0.5, "(no wet)", ha="center"); continue
            qp = np.quantile(pred_wet, quantiles); qt = np.quantile(truth_wet, quantiles)
            ax.loglog(qt, qp, marker=".", linestyle="", color="steelblue", alpha=0.5, markersize=4)
            lo = max(min(qt.min(), qp.min()), 0.1); hi = max(qt.max(), qp.max())
            ax.loglog([lo, hi], [lo, hi], "k--", linewidth=1, label="1:1")
            id_tag = "ID" if gcm == "ACCESS-CM2" else "OOD"
            ax.set_title(f"{DISPLAY[variant]} / {gcm} ({id_tag})", fontsize=10)
            ax.set_xlabel("Truth quantile (mm/jour)")
            ax.set_ylabel("Pred quantile (mm/jour)")
            ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=7)
    plt.suptitle("Q-Q plots queue des intensites (jours pluvieux >1mm)", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(PHASE9_DIR / "01_qq_tail.png", dpi=120, bbox_inches="tight"); plt.close()
    print("[OK] Figure 1 : Q-Q tail")

    # === 2. Return-period Gumbel sur max domain-wide ====================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for c, gcm in enumerate(["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]):
        ax = axes[c]
        for variant, color in [("V5", "steelblue"), ("Noncausal", "orange")]:
            run_label = f"{variant}_{gcm}"
            if run_label not in phase7_arrays: continue
            d = phase7_arrays[run_label]
            pred_mm = np.expm1(np.clip(d["pred_mean"].astype(np.float64), 0, None))
            pred_max_t = np.nanmax(pred_mm.reshape(pred_mm.shape[0], -1), axis=1)
            ps = np.sort(pred_max_t); n = len(ps)
            F = np.arange(1, n + 1) / (n + 1)
            rp = -np.log(-np.log(F))
            ax.plot(rp, ps, color=color, marker=".", linestyle="-",
                     label=f"{DISPLAY[variant]} pred", alpha=0.8, markersize=3)
        d_truth = phase7_arrays.get(f"V5_{gcm}") or phase7_arrays.get(f"Noncausal_{gcm}")
        if d_truth is not None:
            truth_mm = np.expm1(np.clip(d_truth["truth"].astype(np.float64), 0, None))
            tmax = np.nanmax(truth_mm.reshape(truth_mm.shape[0], -1), axis=1)
            ts = np.sort(tmax); F2 = np.arange(1, len(ts) + 1) / (len(ts) + 1)
            rp2 = -np.log(-np.log(F2))
            ax.plot(rp2, ts, color="black", marker=".", linestyle="-",
                     label="Truth", linewidth=2, markersize=3)
        id_tag = "ID" if gcm == "ACCESS-CM2" else "OOD"
        ax.set_xlabel("Gumbel reduced variate")
        ax.set_ylabel("Max precip domain (mm/jour)")
        ax.set_title(f"{gcm} ({id_tag})", fontsize=10)
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.suptitle("Return-period (Gumbel) — daily domain-max precip", fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(PHASE9_DIR / "02_return_period.png", dpi=120, bbox_inches="tight"); plt.close()
    print("[OK] Figure 2 : Return-period Gumbel")

    # === 3. Reliability diagrams (calibration) ==========================
    try:
        from sklearn.calibration import calibration_curve
    except ImportError:
        calibration_curve = None
        print("[WARN] sklearn non installe — reliability skippe")

    if calibration_curve is not None:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for c, thr in enumerate([1.0, 10.0, 50.0]):
            ax = axes[c]
            for variant, color in [("V5", "steelblue"), ("Noncausal", "orange")]:
                obs_all, pred_all = [], []
                for gcm in ["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]:
                    run_label = f"{variant}_{gcm}"
                    if run_label not in phase7_arrays: continue
                    d = phase7_arrays[run_label]
                    ens_mm = np.expm1(np.clip(d["ens"].astype(np.float64), 0, None))
                    truth_mm = np.expm1(np.clip(d["truth"].astype(np.float64), 0, None))
                    fp = (ens_mm > thr).mean(axis=0).flatten()
                    obs = (truth_mm > thr).astype(int).flatten()
                    rng = np.random.default_rng(42)
                    if fp.size > 500_000:
                        idx = rng.choice(fp.size, 500_000, replace=False)
                        fp = fp[idx]; obs = obs[idx]
                    obs_all.append(obs); pred_all.append(fp)
                if not obs_all: continue
                obs_arr = np.concatenate(obs_all); pred_arr = np.concatenate(pred_all)
                if obs_arr.sum() < 10:
                    continue  # trop peu d'evenements positifs
                try:
                    fp_, mp_ = calibration_curve(obs_arr, pred_arr, n_bins=10, strategy="uniform")
                    ax.plot(mp_, fp_, marker="o", color=color, label=DISPLAY[variant], linewidth=2)
                except Exception as e:
                    print(f"  [WARN] calib_curve thr={thr} {variant}: {e}")
            ax.plot([0, 1], [0, 1], "k--", label="perfect")
            ax.set_xlabel("Forecast probability")
            ax.set_ylabel("Observed frequency")
            ax.set_title(f"P(precip > {thr:.0f} mm/jour)")
            ax.legend(); ax.grid(alpha=0.3)
        plt.suptitle("Reliability diagrams (wet-day calibration)", fontsize=12, y=1.01)
        plt.tight_layout()
        plt.savefig(PHASE9_DIR / "03_reliability.png", dpi=120, bbox_inches="tight"); plt.close()
        print("[OK] Figure 3 : Reliability")

    # === 4. Fractions Skill Score vs scale ==============================
    try:
        from scipy.ndimage import uniform_filter
    except ImportError:
        uniform_filter = None

    if uniform_filter is not None:
        def _fss(p_bin, t_bin, n):
            Pf = uniform_filter(p_bin.astype(float), size=n)
            Tf = uniform_filter(t_bin.astype(float), size=n)
            num = ((Pf - Tf) ** 2).mean()
            denom = (Pf ** 2).mean() + (Tf ** 2).mean()
            return 1.0 - num / max(denom, 1e-12)

        scales = [1, 3, 5, 11, 21, 41]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for c, thr in enumerate([1.0, 10.0, 50.0]):
            ax = axes[c]
            for variant, color in [("V5", "steelblue"), ("Noncausal", "orange")]:
                fss_by_scale = []
                for n in scales:
                    vals = []
                    for gcm in ["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]:
                        run_label = f"{variant}_{gcm}"
                        if run_label not in phase7_arrays: continue
                        d = phase7_arrays[run_label]
                        pred_mm = np.expm1(np.clip(d["pred_mean"].astype(np.float64), 0, None))
                        truth_mm = np.expm1(np.clip(d["truth"].astype(np.float64), 0, None))
                        rng = np.random.default_rng(42)
                        n_days = pred_mm.shape[0]
                        idx = rng.choice(n_days, min(30, n_days), replace=False)
                        for t in idx:
                            v = _fss((pred_mm[t] > thr), (truth_mm[t] > thr), n)
                            if np.isfinite(v): vals.append(v)
                    fss_by_scale.append(np.mean(vals) if vals else np.nan)
                ax.plot(scales, fss_by_scale, marker="o", color=color, label=DISPLAY[variant])
            ax.set_xlabel("Neighborhood window (pixels)")
            ax.set_ylabel("FSS")
            ax.set_title(f"thr = {thr:.0f} mm/jour")
            ax.legend(); ax.grid(alpha=0.3)
            ax.set_ylim(0, 1)
        plt.suptitle("Fractions Skill Score vs scale", fontsize=12, y=1.01)
        plt.tight_layout()
        plt.savefig(PHASE9_DIR / "04_fss_vs_scale.png", dpi=120, bbox_inches="tight"); plt.close()
        print("[OK] Figure 4 : FSS vs scale")

    print(f"\n[OK] Phase 9 — figures dans {PHASE9_DIR}")


---

## Phase 10 — Cartes spatiales des biais d'indices d'extremes

Quatre cartes 2D par GCM × variant : **RX1day, CDD, R95p, R10mm** (definitions ETCCDI/WMO).

Standard IPCC AR6 Ch. 11. La Phase 7 livre seulement les moyennes globales — ici on visualise la *structure spatiale* du biais.


In [ ]:
# Phase 10 — Cartes spatiales des biais d'indices d'extremes
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

PHASE10_DIR = RESULTS_DIR / "phase10_extreme_bias_maps"
PHASE10_DIR.mkdir(parents=True, exist_ok=True)

# Display names pour figures (data structures gardent "Noncausal")
DISPLAY = {"V5": "ORACLE", "Noncausal": "CorrDiff"}

if "phase7_arrays" not in dir() or not phase7_arrays:
    print("[ERREUR] phase7_arrays non disponible.")
else:
    def _indices_per_pixel(pr):
        """pr: (T, H, W) en mm/jour. Returns dict de (H, W) maps."""
        T = pr.shape[0]
        rx1day = np.nanmax(pr, axis=0)
        r10mm = (pr >= 10.0).sum(axis=0).astype(np.float32)
        wet = pr[pr >= 1.0]
        thr95 = float(np.percentile(wet, 95)) if wet.size > 0 else 0.0
        r95p = np.where(pr >= thr95, pr, 0.0).sum(axis=0).astype(np.float32)
        dry = (pr < 1.0).astype(np.int8)
        cdd = np.zeros((pr.shape[1], pr.shape[2]), dtype=np.float32)
        run = np.zeros_like(cdd)
        for t in range(T):
            run = np.where(dry[t] == 1, run + 1, 0)
            cdd = np.maximum(cdd, run)
        return {"RX1day": rx1day, "R10mm": r10mm, "R95p": r95p, "CDD": cdd}

    indices_names = ["RX1day", "CDD", "R10mm", "R95p"]
    units = {"RX1day": "mm/jour", "CDD": "jours", "R10mm": "jours", "R95p": "mm"}

    for variant in ["V5", "Noncausal"]:
        fig, axes = plt.subplots(4, 3, figsize=(13, 16))
        any_data = False
        for r_idx, idx_name in enumerate(indices_names):
            for c_idx, gcm in enumerate(["ACCESS-CM2", "EC-Earth3", "NorESM2-MM"]):
                ax = axes[r_idx, c_idx]
                run_label = f"{variant}_{gcm}"
                if run_label not in phase7_arrays:
                    ax.text(0.5, 0.5, "(no data)", ha="center", va="center", transform=ax.transAxes)
                    ax.set_xticks([]); ax.set_yticks([]); continue
                d = phase7_arrays[run_label]
                pred_mm = np.expm1(np.clip(d["pred_mean"].astype(np.float64), 0, None))
                truth_mm = np.expm1(np.clip(d["truth"].astype(np.float64), 0, None))
                ip = _indices_per_pixel(pred_mm); it = _indices_per_pixel(truth_mm)
                bias = ip[idx_name] - it[idx_name]
                vmax = max(np.nanmax(np.abs(bias)), 1e-6)
                im = ax.imshow(bias, cmap="RdBu_r",
                                norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))
                id_tag = "ID" if gcm == "ACCESS-CM2" else "OOD"
                ax.set_title(f"{idx_name} — {gcm} ({id_tag})\n"
                              f"mean bias = {np.nanmean(bias):+.2f} {units[idx_name]}",
                              fontsize=9)
                plt.colorbar(im, ax=ax, shrink=0.7)
                ax.set_xticks([]); ax.set_yticks([])
                any_data = True
        if any_data:
            plt.suptitle(f"Cartes de biais des indices d'extremes — {DISPLAY[variant]}",
                          fontsize=13, y=1.0)
            plt.tight_layout()
            plt.savefig(PHASE10_DIR / f"extreme_bias_{DISPLAY[variant]}.png", dpi=120,
                        bbox_inches="tight")
            print(f"[OK] Figure {DISPLAY[variant]} : 4 indices x 3 GCMs")
        plt.close()

    print(f"\n[OK] Phase 10 dans {PHASE10_DIR}")


---

## Phase 11 — Causalite forte : DAG physique + Path-Specific Effects

Au-dela de la heatmap A_dag de Phase 8, deux tests rigoureux :

1. **DAG physique vs appris (Q_phys)** — comparaison du DAG appris a un DAG attendu construit a partir de la physique atmospherique (couplage vertical descendant GP250 → GP500 → GP850 → SP_HR). Mesure la *coherence physique* du DAG appris.
2. **Path-Specific Effects** — pour chaque arete forte du DAG, on zere uniquement cette arete (les autres restent intactes) et on mesure le Δ sur la prediction. Identifie quelles aretes portent vraiment le signal causal.

Si Q_phys est haut + PSE concentre sur les aretes physiquement attendues → le DAG appris **fonctionne comme un SCM**, pas comme une regularisation cosmetique.


In [ ]:
# Phase 11 — DAG physique + Path-Specific Effects
import numpy as np
import matplotlib.pyplot as plt
import torch
import json

PHASE11_DIR = RESULTS_DIR / "phase11_causal_advanced"
PHASE11_DIR.mkdir(parents=True, exist_ok=True)

if stack_v5["A_dag"] is None:
    print("[SKIP] A_dag indisponible.")
else:
    A_learned = stack_v5["A_dag"].numpy().copy()
    np.fill_diagonal(A_learned, 0)
    n_vars = A_learned.shape[0]
    var_labels = ["GP850_spat", "GP850->GP500", "GP500_spat",
                  "GP500->GP250", "GP250_spat", "SP_HR"][:n_vars]

    # === 1. DAG physique attendu (couplage vertical descendant + meta-paths) ===
    G_phys = np.zeros((n_vars, n_vars))
    L2I = {l: i for i, l in enumerate(var_labels)}
    def add(src, tgt, s):
        if src in L2I and tgt in L2I:
            G_phys[L2I[src], L2I[tgt]] = s
    add("GP250_spat", "GP500_spat", +1)
    add("GP500_spat", "GP850_spat", +1)
    add("GP850_spat", "SP_HR", +1)
    add("GP850->GP500", "GP500_spat", +1)
    add("GP500->GP250", "GP250_spat", +1)

    A_sign = np.sign(A_learned); G_sign = np.sign(G_phys)
    mask = G_sign != 0
    matches = int(((A_sign == G_sign) & mask).sum())
    n_phys = int(mask.sum())
    Q_phys = matches / max(n_phys, 1)
    n_extra = int(((G_sign == 0) & (np.abs(A_learned) > 0.05)).sum())
    print(f"Q_phys = {Q_phys:.3f}  ({matches}/{n_phys} signes attendus corrects)")
    print(f"Aretes 'extra' apprises non prevues par G_phys : {n_extra}")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    im0 = axes[0].imshow(A_learned, cmap="RdBu_r",
                          vmin=-np.abs(A_learned).max(), vmax=np.abs(A_learned).max())
    axes[0].set_title("DAG appris (ORACLE)"); plt.colorbar(im0, ax=axes[0], shrink=0.7)
    im1 = axes[1].imshow(G_phys, cmap="RdBu_r", vmin=-1.5, vmax=1.5)
    axes[1].set_title("DAG physique attendu"); plt.colorbar(im1, ax=axes[1], shrink=0.7)
    conf = np.zeros_like(A_learned)
    conf[(A_sign == G_sign) & mask] = +1
    conf[(A_sign != G_sign) & mask] = -1
    conf[(G_sign == 0) & (np.abs(A_learned) > 0.05)] = -0.4
    im2 = axes[2].imshow(conf, cmap="RdYlGn", vmin=-1.5, vmax=1.5)
    axes[2].set_title(f"Confusion (Q_phys = {Q_phys:.2f})\nvert=correct  rouge=signe inverse  orange=extra")
    plt.colorbar(im2, ax=axes[2], shrink=0.7)
    for a in axes:
        a.set_xticks(range(n_vars)); a.set_xticklabels(var_labels, rotation=45, ha="right", fontsize=8)
        a.set_yticks(range(n_vars)); a.set_yticklabels(var_labels, fontsize=8)
    plt.suptitle("Comparaison DAG appris vs DAG physique", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(PHASE11_DIR / "01_dag_physical_comparison.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("[OK] Figure 1 : DAG appris vs physique")

    # === 2. Path-Specific Effects via edge-masking ======================
    rcn_cell = stack_v5["rcn_runner"].cell
    A_orig = rcn_cell.A_dag.detach().clone()

    edge_strength = np.abs(A_learned)
    edges = [(i, j, edge_strength[i, j]) for i in range(n_vars) for j in range(n_vars)
             if i != j and edge_strength[i, j] > 0.05]
    edges.sort(key=lambda x: -x[2])
    K_edges = min(10, len(edges))
    print(f"PSE sur les {K_edges} aretes les plus fortes...")

    sample_pse = next(iter(test_dataset))
    batch_pse = convert_sample_to_batch(sample_pse, builder, DEVICE)
    with torch.no_grad():
        pred_full = predict_with_stack(stack_v5, batch_pse, K=2, n_steps=18).nanmean(0).squeeze().numpy()

    pse_results = []
    for (i, j, strength) in edges[:K_edges]:
        rcn_cell.A_dag.data.copy_(A_orig)
        rcn_cell.A_dag.data[i, j] = 0.0
        with torch.no_grad():
            pred_cut = predict_with_stack(stack_v5, batch_pse, K=2, n_steps=18).nanmean(0).squeeze().numpy()
        delta = pred_full - pred_cut
        is_phys = bool(G_phys[i, j] != 0)
        pse_results.append({
            "src": var_labels[i], "tgt": var_labels[j],
            "edge_strength": float(strength),
            "pse_magnitude": float(np.abs(delta).mean()),
            "pse_mean": float(np.nanmean(delta)),
            "is_physical_edge": is_phys,
        })
    rcn_cell.A_dag.data.copy_(A_orig)

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    labels = [f"{r['src']}\n→{r['tgt']}" for r in pse_results]
    pse_mag = [r["pse_magnitude"] for r in pse_results]
    bar_colors = ["mediumseagreen" if r["is_physical_edge"] else "steelblue" for r in pse_results]
    ax.bar(range(len(labels)), pse_mag, color=bar_colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("|Delta_pred| moyen (mm/jour log1p)")
    ax.set_title(f"Path-Specific Effects — {K_edges} aretes plus fortes\n"
                  "vert = arete dans G_phys, bleu = arete additionnelle apprise")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(PHASE11_DIR / "02_path_specific_effects.png", dpi=120, bbox_inches="tight")
    plt.close()
    print(f"[OK] Figure 2 : PSE sur {K_edges} aretes")

    # === Save JSON ======================================================
    out = {
        "Q_phys": float(Q_phys),
        "n_physical_edges": n_phys, "n_correct_sign": matches,
        "n_extra_learned_above_0.05": n_extra,
        "G_phys": G_phys.tolist(), "A_learned": A_learned.tolist(),
        "var_labels": var_labels,
        "path_specific_effects": pse_results,
    }
    (PHASE11_DIR / "causal_advanced_results.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"[OK] JSON : {PHASE11_DIR / 'causal_advanced_results.json'}")


---

## Phase 12 — Attribution rigoureuse : Integrated Gradients

Integrated Gradients (Sundararajan 2017) : sensibilite *path-integrated* de la prediction par rapport au LR, depuis une *baseline climatologique* (et non zero, qui est non-physique). Satisfait l'axiome de completude : la somme des attributions egale f(x) − f(baseline).

Protocole Mamalakis 2022 ("Carefully Choose the Baseline") :
- Baseline = moyenne LR sur ~30 jours du test set
- N_steps = 32 pour la quadrature de Riemann
- Implementation manuelle (pas de dependance externe Captum)

Compare Oracle vs CorrDiff : ratio > 1 sur une variable = Oracle l'exploite davantage que la baseline.


In [ ]:
# Phase 12 — Integrated Gradients (manual, Mamalakis 2022 protocol)
import numpy as np
import matplotlib.pyplot as plt
import torch
import json

PHASE12_DIR = RESULTS_DIR / "phase12_integrated_gradients"
PHASE12_DIR.mkdir(parents=True, exist_ok=True)


def _forward_scalar(stack, batch):
    """Scalar output (mean abs of mu_HR) pour IG."""
    enc, rcn, rh, skip = stack["encoder"], stack["rcn_runner"], stack["regression_head"], stack["skip_block"]
    lr = batch["lr"]
    H_init = enc.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr[t] for t in range(lr.shape[0])]
    seq = rcn.run(H_init, drivers, reconstruction_sources=None)
    mu_c = rh(seq.states[-1])
    tshape = batch["residual"][-1].to(DEVICE).shape
    if tshape[-2:] != mu_c.shape[-2:]:
        mu_c = torch.nn.functional.interpolate(mu_c, size=tshape[-2:], mode="bilinear", align_corners=False)
    if skip is not None:
        lr_last = drivers[-1] if drivers[-1].dim() == 4 else drivers[-1].unsqueeze(0)
        mu, _ = skip(lr_last, mu_c)
    else:
        mu = mu_c
    return mu.abs().mean()


def integrated_gradients(stack, batch, baseline_lr, n_steps=32):
    """Sundararajan 2017 : IG = (x - baseline) * mean_alpha grad f(baseline + alpha*(x-baseline))."""
    # batch["lr"] est sur CPU par defaut ; on force tout sur DEVICE pour aligner avec le modele.
    x = batch["lr"].clone().detach().to(DEVICE)
    baseline = baseline_lr.detach().to(DEVICE)
    alphas = torch.linspace(0.0, 1.0, n_steps, device=DEVICE)
    total_grad = torch.zeros_like(x)
    for a in alphas:
        x_interp = (baseline + a * (x - baseline)).detach().requires_grad_(True)
        b2 = dict(batch); b2["lr"] = x_interp
        y = _forward_scalar(stack, b2)
        g = torch.autograd.grad(y, x_interp, retain_graph=False, create_graph=False)[0]
        total_grad = total_grad + g.detach()
    avg_grad = total_grad / n_steps
    return ((x - baseline) * avg_grad).cpu().numpy()


# === Baseline climato : moyenne LR sur ~30 jours du test set =============
print("Construction de la baseline climato (LR mean sur 30 jours)...")
N_clim = 30
lr_accum = None; count = 0
it_clim = iter(test_dataset)
for k in range(N_clim):
    try:
        s = next(it_clim)
    except StopIteration:
        break
    b = convert_sample_to_batch(s, builder, DEVICE)
    if lr_accum is None:
        lr_accum = b["lr"].detach().clone().cpu()
    else:
        lr_accum = lr_accum + b["lr"].detach().clone().cpu()
    count += 1
baseline_lr = (lr_accum / count).to(DEVICE)
print(f"  baseline : shape={tuple(baseline_lr.shape)}, mean={baseline_lr.mean().item():.4f}, "
      f"std={baseline_lr.std().item():.4f}, n_samples={count}")

# === IG pour ORACLE vs CorrDiff sur le meme sample ==========================
ig_results = {"V5": {}, "Noncausal": {}}
sample_ig = next(iter(test_dataset))
batch_ig = convert_sample_to_batch(sample_ig, builder, DEVICE)
lr_vars = list(CONFIG.data.lr_variables)

for stack_name, stack in [("V5", stack_v5), ("Noncausal", stack_nc)]:
    print(f"  Computing IG for {stack_name} (n_steps=32)...")
    ig = integrated_gradients(stack, batch_ig, baseline_lr, n_steps=32)
    # Aggregate to per-variable score (canal = derniere dim pour graph, ou 2e pour grid)
    if ig.ndim == 3:        # (T, N, C) graphe
        ig_per_var = np.abs(ig).mean(axis=(0, 1))
    elif ig.ndim == 4:      # (T, C, H, W) grid
        ig_per_var = np.abs(ig).mean(axis=(0, 2, 3))
    elif ig.ndim == 5:      # (B, T, C, H, W)
        ig_per_var = np.abs(ig).mean(axis=(0, 1, 3, 4))
    else:
        ig_per_var = np.abs(ig).reshape(ig.shape[-1], -1).mean(axis=-1)
    for i, v in enumerate(lr_vars):
        if i < len(ig_per_var):
            ig_results[stack_name][v] = float(ig_per_var[i])
        else:
            ig_results[stack_name][v] = 0.0

# === Plot ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(lr_vars)); w = 0.35
v5_vals = [ig_results["V5"].get(v, 0) for v in lr_vars]
nc_vals = [ig_results["Noncausal"].get(v, 0) for v in lr_vars]
axes[0].bar(x - w/2, v5_vals, w, label="ORACLE", color="steelblue")
axes[0].bar(x + w/2, nc_vals, w, label="CorrDiff", color="orange")
axes[0].set_xticks(x); axes[0].set_xticklabels(lr_vars, rotation=45, ha="right", fontsize=7)
axes[0].set_ylabel("|IG| moyen par variable")
axes[0].set_title("Integrated Gradients par variable LR (ORACLE vs CorrDiff)\n(baseline = climato moyenne)")
axes[0].legend(); axes[0].grid(alpha=0.3)

ratios = [v5_vals[i] / max(nc_vals[i], 1e-12) for i in range(len(lr_vars))]
axes[1].bar(x, ratios, color="purple", alpha=0.7)
axes[1].axhline(1.0, color="red", linestyle="--", label="ratio=1")
axes[1].set_xticks(x); axes[1].set_xticklabels(lr_vars, rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Ratio IG ORACLE / CorrDiff")
axes[1].set_title("Ratio IG ORACLE / CorrDiff — > 1 = ORACLE exploite davantage cette variable")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PHASE12_DIR / "01_integrated_gradients.png", dpi=120, bbox_inches="tight")
plt.close()
print("[OK] Figure : Integrated Gradients par variable")

(PHASE12_DIR / "ig_results.json").write_text(
    json.dumps(ig_results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[OK] JSON : {PHASE12_DIR / 'ig_results.json'}")


---

## Synthèse — résultats consolidés

Après exécution des Phases 6, 7, 8, les résultats sont dans :
```
RESULTS_DIR/
├── phase6_in_distribution.json
├── phase7_ood_physical_diagnostics.json (ou phase7_ood.json si Mode A)
└── phase8_intervention.json
```

### Construction du verdict pour la soutenance

**Si Oracle gagne in-distribution + OOD + intervention** :
> *« Oracle bat le baseline CorrDiff sur la majorité des métriques in-distribution, dégrade moins sous changement de climat, et satisfait Q_int ≥ 0.9 sur le protocole d'intervention. »*

**Si Oracle gagne seulement sur calibration + structure + intervention** (cas trilemme) :
> *« Oracle affiche un compromis assume : performance in-distribution proche du CorrDiff, calibration probabiliste significativement améliorée (+30 %), fidélité spectrale +10 %, et capacité d'intervention démontrée. »*

**Si Oracle ne gagne nulle part** :
> *Diagnostic montrant que les pertes d'Oracle ont peut-etre été mal calibrées. Plan B : ablation study.*

In [ ]:
# Synthese des resultats sauvegardes
import json

print("=" * 70)
print("SYNTHESE Oracle - fichiers generes")
print("=" * 70)
for fname in ["phase6_in_distribution", "phase7_ood_physical_diagnostics", "phase7_ood", "phase8_intervention"]:
    p = RESULTS_DIR / f"{fname}.json"
    if p.exists():
        size_kb = p.stat().st_size / 1024
        print(f"  [OK] {p.name:50s}  {size_kb:>8.1f} KB")
    else:
        print(f"  [--] {p.name:50s}  (non genere)")

print()
print("Pour le memoire, voir :")
print("  - Tableau metrique x variant      -> phase6_in_distribution.json (cle 'comparison')")
print("  - D_OOD ou diagnostics physiques  -> phase7_*.json")
print("  - Q_int et signe interventions    -> phase8_intervention.json")